In [ ]:
#!/usr/bin/env python
# coding: utf-8
"""
================================================================================
 GC_TAM_Stage1_Figure1.py
 Gastric-cancer TAM atlas  ->  multicellular programmes (MCP)  ->  Figure 1

 Visual template : Motevasseli et al., Acta Neuropathol Commun (2024) 12:133,
                   Figure 1 (panels A-J) -- same panel types, colours, fonts.
 Analytical logic: NO predefined patient groups. Patients are ordered along a
                   data-driven multicellular programme (MCP) learned with MOFA+
                   from NON-MYELOID pseudobulk profiles; myeloid cells are the
                   response variable (no circularity).

 DATA (all verified from GEO series matrices; nothing inferred or synthetic)
   GSE150290 (Kim et al., npj Precis Oncol 2022; 10x 3' v2, hg19, biopsies)
       Pat01-Pat24 "B" = cancer (23 libraries; Pat21 has no cancer library)
       Pat01-Pat24 "A" = adjacent normal  -> inferCNV negative control only
       Pat25-Pat29     = non-cancer controls -> NOT used
   GSE183904 (Kumar et al., Cancer Discov 2022; 10x 5', count CSVs)
       26 primary tumours; 10 primary normals -> inferCNV negative control
       peritoneal samples (sample19/20/37/38) -> NOT used in Figure 1

 STAGES (each checkpointed to disk; re-run resumes)
   S0  sample sheet
   S1  load  -> cell calling (GSE150290 raw droplets, Cell Ranger-2 OrdMag,
               i.e. the algorithm of the original authors' Cell Ranger 2.1.1)
         -> per-sample QC (fixed floors + 5-MAD upper outliers) -> Scrublet
         -> scVI integration (template-paper architecture; 3,000 HVGs; trained on a
            per-sample geometric sketch, latent space inferred for ALL cells)
         -> major-lineage annotation (marker panels, ambiguity flagging)
         -> inferCNV (infercnvpy; immune+stromal reference; threshold from
            adjacent-normal epithelium) -> malignant epithelium
         -> adjacent-normal non-epithelial cells removed (no downstream role)
         -> myeloid re-integration (2,000 HVGs, all myeloid cells) + state annotation
   S2  MOFA+ on non-myeloid pseudobulk (patients x cell types) -> MCP
         + leave-one-dataset-out reproducibility + Hallmark annotation
   FIG1 A UMAPs | B marker UMAPs | C dot plot | D composition along MCP
        E state abundance vs MCP (CLR + OLS, dataset-adjusted, BH-FDR)
        F Hallmark AUCell UMAPs | G,H pseudobulk DESeq2 volcanoes along MCP
        I,J pre-ranked GSEA (Hallmark 2020)
   SUPP S1 QC | S2 integration metrics | S3 inferCNV | S4 MOFA/MCP
        S5 myeloid annotation evidence + per-dataset DE concordance

 RUN:  python GC_TAM_Stage1_Figure1.py        (GPU strongly recommended)
================================================================================
"""

# ==============================================================================
# 0. IMPORTS / DEPENDENCIES
# ==============================================================================
import os
import re
import sys
import gzip
import json
import time
import shutil
import tarfile
import warnings
import subprocess
import importlib
from pathlib import Path


def _ensure(pkg_import, pip_name=None):
    try:
        return importlib.import_module(pkg_import)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or pkg_import], check=True)
        return importlib.import_module(pkg_import)


for _imp, _pip in [("scanpy", "scanpy"), ("anndata", "anndata"), ("scvi", "scvi-tools"),
                   ("infercnvpy", "infercnvpy"), ("mofapy2", "mofapy2"), ("pydeseq2", "pydeseq2"),
                   ("gseapy", "gseapy"), ("statsmodels", "statsmodels"), ("adjustText", "adjustText"),
                   ("h5py", "h5py"), ("sklearn", "scikit-learn"), ("skmisc", "scikit-misc"), ("igraph", "igraph"),
                   ("leidenalg", "leidenalg"), ("geosketch", "geosketch")]:
    _ensure(_imp, _pip)

import numpy as np
import pandas as pd
import scipy.io
import scipy.sparse as sp
from scipy import stats
import scanpy as sc
import anndata as ad
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ==============================================================================
# 1. CONFIGURATION
# ==============================================================================
BASE = Path(os.environ.get("GC_BASE", "/home/arshia/payanname"))
RAW_150290 = BASE / "GSE150290_RAW"
RAW_183904 = BASE / "GSE183904_RAW"
OUT = BASE / "GC_Fig1_outputs"
RES = OUT / "resources"
H5 = OUT / "h5ad"
TAB = OUT / "tables"
FIG_MAIN = OUT / "figures" / "main"
FIG_SUPP = OUT / "figures" / "supp"
TMP = OUT / "tmp_extract"
for _d in [OUT, RES, H5, TAB, FIG_MAIN, FIG_SUPP, TMP]:
    _d.mkdir(parents=True, exist_ok=True)
LOGFILE = OUT / "run_log.txt"

SEED = 0
N_JOBS = max(1, (os.cpu_count() or 2) - 1)

# GENCODE gene coordinates (inferCNV positions; Ensembl->symbol fallback)
GENCODE_URL = ("https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/"
               "release_44/gencode.v44.basic.annotation.gtf.gz")
GENCODE_GTF = RES / "gencode.v44.basic.annotation.gtf.gz"
HALLMARK_CACHE = RES / "MSigDB_Hallmark_2020.json"

# --- cell calling (GSE150290 raw droplet matrices) ---
ORDMAG_EXPECTED_CELLS = 3000        # Cell Ranger 2.x default
ORDMAG_QUANTILE, ORDMAG_RATIO = 0.99, 10.0

# --- QC (template paper: >=200 genes, <25% mito; plus 5-MAD upper outliers) ---
QC_MIN_GENES, QC_MAX_PCT_MT, QC_NMADS = 200, 25.0, 5.0
QC_MIN_CELLS_PER_GENE, QC_MIN_CELLS_PER_SAMPLE = 3, 100

# --- integration (template paper parameters) ---
# HVGs define ONLY the scVI latent space; every downstream quantitative analysis
# (inferCNV, pseudobulk DE, MOFA+, AUCell, dot plots, composition) uses ALL genes.
# 3,000 (atlas) / 2,000 (myeloid) follows the scIB benchmark (Luecken et al., Nat Methods 2022)
# and scvi-tools recommendations; technical and clonotype (V-segment) genes are excluded.
N_HVG = 3000
SCVI_KW = dict(n_latent=20, n_layers=2, gene_likelihood="nb", use_layer_norm="both",
               use_batch_norm="none", encode_covariates=True, dropout_rate=0.2)
SCVI_MAX_EPOCHS = 300
N_NEIGHBORS, LEIDEN_RES_MAJOR = 30, 1.0

# --- cell-number reduction (no cell that feeds a downstream estimate is discarded) ---
# (1) scVI is TRAINED on a per-sample geometric sketch (Hie et al., Cell Syst 2019), which
#     preserves rare populations and balances large vs small samples; the trained model then
#     INFERS the latent space for every cell (all cells are clustered and annotated).
# (2) Adjacent-normal samples serve only as the inferCNV negative control; after inferCNV their
#     non-epithelial cells have no analytical role and are removed from all saved objects.
# (3) Myeloid cells (the object of study) are never sketched or downsampled.
SKETCH_PER_SAMPLE = 2000
SKETCH_PER_SAMPLE_MYELOID = None
PIPE_TAG = "v3"                  # new checkpoint names -> objects from earlier versions are never reused
GENE_TAG = "ensv44"              # S1a/S1b built with Ensembl-ID -> GENCODE v44 symbol harmonisation
MY_BUILD_TAG = "m2"              # myeloid object (scVI + Leiden) -- never rebuilt, cluster IDs frozen
MY_LABEL_TAG = "m3"              # myeloid labels after gene-level review (MANUAL_OVERRIDES_MYELOID)
MCP_TAG = "amb20"                # MCP learned after cross-lineage ambient-gene filtering

# --- annotation ---
ANNOT_MIN_TOP, ANNOT_MARGIN, ANNOT_SECOND_HIGH = 0.30, 0.15, 0.50
MANUAL_OVERRIDES_MAJOR = {}      # {"leiden_major cluster id": "Lineage"} after reviewing tables
# Gene-level review of ambiguous clusters (tables/S1_myeloid_cluster_gene_means.csv).
# Rules: SPP1 mean log-expr > 4 -> TAM_SPP1; VCAN/FCN1/S100A8+ C1QA-low -> classical monocyte;
# C1Q-intermediate, VCAN/FCN1-low, CDKN1C-low (not non-classical monocytes) -> transitional macrophage;
# IL1B/CXCL8/S100A8-high with C1Q -> inflammatory TAM. Expected cluster sizes guard against drift.
MANUAL_OVERRIDES_MYELOID = {
    "8":  ("TAM_SPP1", 144),          # SPP1 4.44, APOE 4.03
    "12": ("TAM_SPP1", 658),          # SPP1 4.89 (highest of all clusters)
    "21": ("TAM_Inflammatory", 299),  # IL1B 3.42, CXCL8 3.83, S100A8 2.85
    "2":  ("TAM_Transitional", 457),  # C1QA 1.90, VCAN 0.35, CDKN1C 0.10
    "22": ("TAM_Transitional", 278),  # C1QA 2.45, VCAN 0.44, CDKN1C 0.07
    "1":  ("Mono_classical", 446),    # C1QA 0.55, VCAN 1.15, FCN1 1.11, EREG 1.80
}
# MOFA views: a gene is kept in view v only if its CPM in lineage v is >= this fraction of its
# maximum CPM in any other lineage (removes ambient/contaminating transcripts, e.g. KRT19/TFF1
# in lymphocyte views or SPP1/CXCL8 from myeloid cells -- the latter also protects against
# circularity, as myeloid cells are the response variable).
AMBIENT_MIN_RATIO = 0.2
# Axis for Figure 1. The pre-specified rule (largest total R2) gave Factor1, which showed no
# association with any myeloid state. A screen of ALL 8 factors x 12 states (96 tests, global BH-FDR,
# sign agreement in both datasets required; tables/S2_factor_screen_*) identified Factor4.
# Factor4 = stromal-vascular programme (fibroblast IL6/JAK/STAT3, TNF/NF-kB, EMT; proliferating,
# glycolytic pericytes vs contractile pericytes and quiescent endothelium). Oriented so that
# POSITIVE = activated stroma (fibroblast TNF/NF-kB genes carry positive weights).
MCP_AXIS_FACTOR = "Factor4"      # None -> pre-specified rule
AXIS_TAG = "F4"
AXIS_LABEL = "SAP"               # stromal activation programme
AXIS_ORIENT_VIEW, AXIS_ORIENT_SET = "Fibroblast", "TNF-alpha Signaling via NF-kB"
CONTAM_RATIO, CONTAM_FOLD = 0.5, 3.0   # doublet cluster: panel >=50% of true lineage AND >=3x myeloid median

# --- inferCNV ---
CNV_WINDOW, CNV_REF_PER_LINEAGE, CNV_NORMAL_Q = 250, 1500, 0.99

# --- myeloid ---
N_HVG_MYELOID, LEIDEN_RES_MYELOID = 2000, 1.5    # 1.5 = template paper resolution

# --- MCP (MOFA+) ---
MCP_VIEWS = ["Malignant", "Fibroblast", "Endothelial", "Pericyte_SMC", "T_NK", "B", "Plasma"]
# MOFA+ models missing views natively; a view enters if >=12 tumours have >=10 cells of that type
PB_MIN_CELLS, MOFA_MIN_SAMPLES_VIEW, MOFA_N_FEATURES = 10, 12, 1000
MOFA_FACTORS, MOFA_ITER = 8, 3000
MCP_R2_ACTIVE = 2.0              # % variance: factor "active" in a view

# --- Figure-1 statistics ---
COMP_MIN_MYELOID = 30            # samples with >=30 myeloid cells enter composition tests
DE_MIN_CELLS, DE_MIN_SAMPLES = 10, 8
DE_FDR, DE_LFC = 0.05, 0.5       # log2FC per 1 SD of MCP
GSEA_PERM = 1000

FORCE = set()                    # e.g. {"load", "qc", "integrate", "cnv", "myeloid", "mofa", "de"}

# ==============================================================================
# 2. VERIFIED SAMPLE METADATA (from GEO series matrices)
# ==============================================================================
# GSE150290 cancer ("B") libraries: Lauren + GEO "diagnosis" field, verbatim.
GSE150290_TUMOR = {
    "Pat01": ("Intestinal", "EGC"), "Pat02": ("Diffuse", "AGC"), "Pat03": ("Diffuse", "EGC"),
    "Pat04": ("Diffuse", "EGC"), "Pat05": ("Diffuse", "EGC"), "Pat06": ("Diffuse", "EGC"),
    "Pat07": ("Intestinal", "EGC"), "Pat08": ("Diffuse", "EGC"), "Pat09": ("Intestinal", "EGC"),
    "Pat10": ("Diffuse", "EGC"), "Pat11": ("Diffuse", "AGC"), "Pat12": ("Diffuse", "AGC"),
    "Pat13": ("Intestinal", "ECG"), "Pat14": ("Diffuse", "AGC"), "Pat15": ("Diffuse", "AGC"),
    "Pat16": ("Diffuse", "EGC"), "Pat17": ("Intestinal", "EGC"), "Pat18": ("Intestinal", "EGC"),
    "Pat19": ("Intestinal", "EGC"), "Pat20": ("Diffuse", "AGC"), "Pat22": ("Intestinal", "EGC"),
    "Pat23": ("Diffuse", "AGC"), "Pat24": ("Diffuse", "AGC"),
}
GSE150290_NORMAL_PATIENTS = [f"Pat{i:02d}" for i in range(1, 25)]      # Pat01-Pat24 "A"

# GSE183904: sampleN -> (GSM, manuscript patient ID, tissue); peritoneal samples excluded.
GSE183904_SAMPLES = {
    "sample1": ("GSM5573466", "NGCII518", "Normal"), "sample2": ("GSM5573467", "NGCII518", "Tumor"),
    "sample3": ("GSM5573468", "NGCII519", "Tumor"), "sample4": ("GSM5573469", "NGCII520", "Normal"),
    "sample5": ("GSM5573470", "NGCII520", "Tumor"), "sample6": ("GSM5573471", "NGCII521", "Normal"),
    "sample7": ("GSM5573472", "NGCII521", "Tumor"), "sample8": ("GSM5573473", "NGCII524", "Tumor"),
    "sample9": ("GSM5573474", "NGCII525", "Normal"), "sample10": ("GSM5573475", "NGCII525", "Tumor"),
    "sample11": ("GSM5573476", "NGCII527", "Normal"), "sample12": ("GSM5573477", "NGCII527", "Tumor"),
    "sample13": ("GSM5573478", "NGCII529", "Tumor"), "sample14": ("GSM5573479", "NGCII502", "Tumor"),
    "sample15": ("GSM5573480", "NGCII511", "Tumor"), "sample16": ("GSM5573481", "NGCII499", "Tumor"),
    "sample17": ("GSM5573482", "NGCII509", "Tumor"), "sample18": ("GSM5573483", "NGCII498", "Tumor"),
    "sample21": ("GSM5573486", "NGCII513", "Normal"), "sample22": ("GSM5573487", "NGCII513", "Tumor"),
    "sample23": ("GSM5573488", "NGCII514", "Normal"), "sample24": ("GSM5573489", "NGCII514", "Tumor"),
    "sample25": ("GSM5573490", "NGCII512", "Normal"), "sample26": ("GSM5573491", "NGCII512", "Tumor"),
    "sample27": ("GSM5573492", "NGCII510", "Tumor"), "sample28": ("GSM5573493", "NGCII531", "Tumor"),
    "sample29": ("GSM5573494", "NGCII522", "Tumor"), "sample30": ("GSM5573495", "NGCII533", "Tumor"),
    "sample31": ("GSM5573496", "NGCII536", "Normal"), "sample32": ("GSM5573497", "NGCII539", "Tumor"),
    "sample33": ("GSM5573498", "NGCII540", "Tumor"), "sample34": ("GSM5573499", "NGCII541", "Tumor"),
    "sample35": ("GSM5573500", "NGCII538", "Normal"), "sample36": ("GSM5573501", "NGCII538", "Tumor"),
    "sample39": ("GSM5573504", "NGCII543", "Tumor"), "sample40": ("GSM5573505", "NGCII545", "Tumor"),
}

# ==============================================================================
# 3. MARKER PANELS
# ==============================================================================
MAJOR_PANELS = {
    "Epithelial":   ["EPCAM", "KRT8", "KRT18", "KRT19", "CLDN4", "CLDN7", "ELF3"],
    "T_NK":         ["CD3D", "CD3E", "CD2", "TRAC", "NKG7", "GNLY", "KLRD1"],
    "B":            ["MS4A1", "CD79A", "CD19", "BANK1", "CD22"],
    "Plasma":       ["MZB1", "JCHAIN", "SDC1", "TNFRSF17", "DERL3"],
    "Myeloid":      ["CD14", "LYZ", "CD68", "AIF1", "CSF1R", "FCER1G", "TYROBP"],
    "Mast":         ["TPSAB1", "TPSB2", "CPA3", "KIT", "MS4A2"],
    "Fibroblast":   ["DCN", "LUM", "COL1A1", "COL1A2", "PDGFRA", "COL3A1"],
    "Pericyte_SMC": ["RGS5", "ACTA2", "MYH11", "PDGFRB", "NOTCH3", "DES"],
    "Endothelial":  ["PECAM1", "VWF", "CDH5", "PLVAP", "CLDN5", "RAMP2"],
    "Erythrocyte":  ["HBB", "HBA1", "HBA2"],
}
MYELOID_PANELS = {
    # pan-myeloid genes (CD14, LYZ, APOE, APOC1, TREM2, TUBB, STMN1) removed: they are shared by
    # several states and made cluster labels ambiguous
    "Mono_classical":    ["FCN1", "VCAN", "S100A8", "S100A9", "S100A12", "CD300E"],
    "Mono_nonclassical": ["FCGR3A", "CDKN1C", "HES4", "TCF7L2", "LILRB2"],
    "TAM_SPP1":          ["SPP1", "GPNMB", "FBP1", "MMP9", "MMP12", "CHI3L1"],
    "TAM_C1QC":          ["C1QA", "C1QB", "C1QC", "CCL18", "MS4A6A"],
    "TAM_FOLR2":         ["FOLR2", "LYVE1", "SELENOP", "F13A1", "MRC1", "STAB1"],
    "TAM_Inflammatory":  ["IL1B", "CXCL8", "CCL3", "CCL4", "EREG", "NLRP3", "TNF"],
    "TAM_IFN":           ["ISG15", "IFIT1", "IFIT3", "IFI6", "MX1", "CXCL10"],
    "TAM_Prolif":        ["MKI67", "TOP2A", "CENPF", "UBE2C", "BIRC5"],
    "cDC1":              ["CLEC9A", "XCR1", "IDO1", "CADM1"],
    "cDC2":              ["CD1C", "FCER1A", "CLEC10A", "CD1E"],
    "mregDC":            ["LAMP3", "CCR7", "FSCN1", "CCL19", "CCL22"],
    "pDC":               ["LILRA4", "IL3RA", "CLEC4C", "IRF7", "GZMB"],
    "Neutrophil":        ["CSF3R", "FCGR3B", "CXCR2", "CMTM2", "G0S2"],
}
# Figure-1G/H compartments (analogue of BMD-TAM vs MG-TAM in the template).
COMPARTMENTS = {
    "Monocyte-derived TAM": ["TAM_SPP1", "TAM_Inflammatory", "TAM_IFN"],
    "Resident-like TAM":    ["TAM_C1QC", "TAM_FOLR2"],
}
MYELOID_ORDER = ["Mono_classical", "Mono_nonclassical", "TAM_Transitional", "TAM_Inflammatory",
                 "TAM_SPP1", "TAM_C1QC", "TAM_FOLR2", "TAM_IFN", "TAM_Prolif",
                 "cDC1", "cDC2", "mregDC", "pDC", "Neutrophil"]
EXTRA_MARKERS = {"TAM_Transitional": ["HLA-DRA", "CD74", "CD14"]}
FIG1B_MARKERS = ["C1QA", "SPP1", "FOLR2", "FCN1", "IL1B", "ISG15", "CD1C", "MKI67"]
FIG1F_HALLMARKS = ["Hypoxia", "Glycolysis", "Epithelial Mesenchymal Transition",
                   "Oxidative Phosphorylation", "Cholesterol Homeostasis", "G2-M Checkpoint",
                   "Inflammatory Response", "Interferon Gamma Response",
                   "TNF-alpha Signaling via NF-kB"]

# ==============================================================================
# 4. FIGURE STYLE (template: Arial-like sans, small fonts, thin lines)
# ==============================================================================
def _pick_font():
    from matplotlib import font_manager
    avail = {f.name for f in font_manager.fontManager.ttflist}
    for f in ["Arial", "Helvetica", "Liberation Sans", "Nimbus Sans", "DejaVu Sans"]:
        if f in avail:
            return f
    return "DejaVu Sans"


FONT = _pick_font()
mpl.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": [FONT],
    "font.size": 7, "axes.titlesize": 8, "axes.titleweight": "bold",
    "axes.labelsize": 7, "xtick.labelsize": 6, "ytick.labelsize": 6,
    "legend.fontsize": 6, "legend.title_fontsize": 6.5,
    "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "xtick.major.size": 2.5, "ytick.major.size": 2.5, "lines.linewidth": 0.8,
    "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",
    "savefig.dpi": 600, "figure.dpi": 150, "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
})
sc.settings.verbosity = 1
sc.settings.n_jobs = N_JOBS

LINEAGE_PALETTE = {
    "Malignant": "#1F4E9C", "Epithelial_NM": "#8FB3E0", "Myeloid": "#F28E2B",
    "T_NK": "#D62728", "B": "#3A89C9", "Plasma": "#8E44AD", "Mast": "#8C564B",
    "Fibroblast": "#E377C2", "Pericyte_SMC": "#7F9C3B", "Endothelial": "#E6A117",
}
MYELOID_PALETTE = {
    "TAM_C1QC": "#6A3D9A", "TAM_FOLR2": "#33A02C", "TAM_SPP1": "#E31A1C",
    "TAM_Inflammatory": "#FF9E4A", "TAM_IFN": "#F2D400", "TAM_Prolif": "#1A1A1A",
    "TAM_Transitional": "#A6761D",
    "Mono_classical": "#8C2D19", "Mono_nonclassical": "#C98B63", "cDC1": "#1F5FA8",
    "cDC2": "#4F8FD6", "mregDC": "#9ECAE9", "pDC": "#8C8C8C", "Neutrophil": "#BCBD22",
}
DATASET_PALETTE = {"GSE183904": "#4A7EB5", "GSE150290": "#E3893F"}
HALLMARK_CMAP = LinearSegmentedColormap.from_list(
    "hallmark", ["#FFFBEA", "#FDD5A0", "#F07C6C", "#B8337A", "#4B0C5E"])
MARKER_CMAP = "viridis_r"


def log(msg):
    line = f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
    print(line, flush=True)
    with open(LOGFILE, "a") as fh:
        fh.write(line + "\n")


def savefig(fig, name, folder=FIG_MAIN):
    for ext in ["pdf", "svg", "png"]:
        fig.savefig(folder / f"{name}.{ext}")
    plt.close(fig)
    log(f"  saved figure {folder.name}/{name}")


def panel_label(fig_or_ax, letter, x=-0.02, y=1.02):
    if isinstance(fig_or_ax, plt.Figure):
        fig_or_ax.text(0.0, 1.0, letter, fontsize=10, fontweight="bold", va="bottom", ha="left",
                       transform=fig_or_ax.transFigure)
    else:
        fig_or_ax.text(x, y, letter, fontsize=10, fontweight="bold", va="bottom", ha="right",
                       transform=fig_or_ax.transAxes)


def rasterize_points(ax):
    for coll in ax.collections:
        coll.set_rasterized(True)


def umap_axis_arrows(ax, label="UMAP"):
    """Template-style L-shaped axis arrows in the lower-left corner."""
    kw = dict(xycoords="axes fraction", textcoords="axes fraction",
              arrowprops=dict(arrowstyle="-|>", lw=0.6, color="black", mutation_scale=6))
    ax.annotate("", xy=(0.16, 0.02), xytext=(0.02, 0.02), **kw)
    ax.annotate("", xy=(0.02, 0.16), xytext=(0.02, 0.02), **kw)
    ax.text(0.02, -0.02, f"{label} 1", transform=ax.transAxes, fontsize=5.5, va="top", ha="left")
    ax.text(-0.01, 0.02, f"{label} 2", transform=ax.transAxes, fontsize=5.5, va="bottom",
            ha="right", rotation=90)


def set_palette(adata, key, palette):
    adata.obs[key] = adata.obs[key].astype("category")
    adata.uns[f"{key}_colors"] = [palette.get(c, "#BBBBBB") for c in adata.obs[key].cat.categories]


def fdr_stars(q):
    return "***" if q < 0.001 else "**" if q < 0.01 else "*" if q < 0.05 else "ns"


# ==============================================================================
# 5. GENERIC UTILITIES
# ==============================================================================
HOUSEKEEPING = {"ACTB", "GAPDH", "B2M", "MALAT1", "TMSB4X", "EEF1A1", "FTL", "RPLP0"}
# technical genes + immunoglobulin / TCR variable segments (clonotype-driven, not cell state)
EXCLUDE_GENE_RE = re.compile(r"^(MT-|RP[SL]\d|HB[ABDEGMQZ]\d?$|MTRNR|IG[HKL]V|TR[ABDG]V)")


def aggregate_duplicate_genes(X, names):
    """Sum columns sharing a gene symbol (cells x genes, CSR)."""
    names = pd.Index(pd.Series(names).astype(str))
    if names.is_unique:
        return X.tocsr(), names
    codes, uniq = pd.factorize(names)
    agg = sp.csr_matrix((np.ones(len(codes), dtype=np.float32), (np.arange(len(codes)), codes)),
                        shape=(len(codes), len(uniq)))
    return (X.tocsr() @ agg).tocsr(), pd.Index(uniq)


def ordmag_cell_calling(total_counts, expected=ORDMAG_EXPECTED_CELLS,
                        q=ORDMAG_QUANTILE, ratio=ORDMAG_RATIO):
    """Cell Ranger 2.x 'order-of-magnitude' barcode filter (Zheng et al., 2017)."""
    s = np.sort(np.asarray(total_counts))[::-1]
    top = s[:min(expected, len(s))]
    top = top[top > 0]
    thr = max(float(np.quantile(top, q)) / ratio, 1.0)
    return np.asarray(total_counts) >= thr, thr


def read_10x_v2_tar(tar_path):
    """Read a Cell Ranger 2.x raw_gene_bc_matrices tarball -> (cells x genes CSR, barcodes, symbols)."""
    dest = TMP / Path(tar_path).name.replace(".tar.gz", "")
    if dest.exists():
        shutil.rmtree(dest)
    dest.mkdir(parents=True)
    with tarfile.open(tar_path, "r:*") as tf:
        try:
            tf.extractall(dest, filter="data")
        except TypeError:
            tf.extractall(dest)
    mtx = next((p for p in dest.rglob("matrix.mtx*")), None)
    if mtx is None:
        raise FileNotFoundError(f"No matrix.mtx inside {tar_path}")
    d = mtx.parent
    gfile = next((p for p in d.iterdir() if p.name.startswith(("genes.tsv", "features.tsv"))), None)
    bfile = next((p for p in d.iterdir() if p.name.startswith("barcodes.tsv")), None)
    M = scipy.io.mmread(str(mtx)).T.tocsr().astype(np.float32)
    genes = pd.read_csv(gfile, sep="\t", header=None)
    symbols = (genes.iloc[:, 1] if genes.shape[1] > 1 else genes.iloc[:, 0]).astype(str).values
    gene_ids = genes.iloc[:, 0].astype(str).str.split(".").str[0].values
    barcodes = pd.read_csv(bfile, header=None).iloc[:, 0].astype(str).values
    shutil.rmtree(dest)
    return M, barcodes, symbols, gene_ids


def _looks_like_genes(values):
    v = pd.Index(pd.Series(values).astype(str))
    return len(HOUSEKEEPING & set(v)) >= 2 or np.mean(v.str.startswith("ENSG")) > 0.5


def read_count_csv(path, id2name=None, chunksize=1000):
    """GEO count CSV -> (cells x genes CSR, cells, genes). Orientation auto-detected and verified.
    Read positionally (header=None) so blank/duplicated header names cannot break dtype parsing."""
    header = pd.read_csv(path, nrows=0).columns
    col_names = np.asarray(header[1:]).astype(str)
    first_col = pd.read_csv(path, usecols=[0], dtype=str).iloc[:, 0]
    if _looks_like_genes(first_col):
        orient = "genes_x_cells"
    elif _looks_like_genes(col_names):
        orient = "cells_x_genes"
    else:
        raise ValueError(f"Cannot determine orientation of {path}")
    ncol = len(header)
    dtype = {0: str, **{i: np.float32 for i in range(1, ncol)}}
    rows, blocks = [], []
    for ch in pd.read_csv(path, header=None, skiprows=1, dtype=dtype, chunksize=chunksize):
        rows.extend(ch.iloc[:, 0].astype(str).tolist())
        blocks.append(sp.csr_matrix(ch.iloc[:, 1:].to_numpy(dtype=np.float32)))
    M = sp.vstack(blocks).tocsr()
    if orient == "genes_x_cells":
        X, cells, genes = M.T.tocsr(), col_names, np.asarray(rows)
    else:
        X, cells, genes = M, np.asarray(rows), col_names
    probe = X.data[:200000]
    if probe.size and np.mean(np.abs(probe - np.round(probe)) > 1e-6) > 0:
        raise ValueError(f"{path}: values are not integer counts")
    genes = pd.Index(genes)
    if np.mean(genes.str.startswith("ENSG")) > 0.5:
        if id2name is None:
            raise ValueError("Ensembl IDs found but no GTF mapping supplied")
        genes = pd.Index([id2name.get(g.split(".")[0], g) for g in genes])
    return X, cells, genes.values


def load_gencode():
    if not GENCODE_GTF.exists():
        log(f"Downloading GENCODE v44 basic GTF -> {GENCODE_GTF}")
        import urllib.request
        urllib.request.urlretrieve(GENCODE_URL, GENCODE_GTF)
    cache = RES / "gencode_v44_genes.tsv"
    if cache.exists():
        return pd.read_csv(cache, sep="\t")
    rows = []
    with gzip.open(GENCODE_GTF, "rt") as fh:
        for line in fh:
            if line.startswith("#"):
                continue
            p = line.split("\t", 8)
            if p[2] != "gene":
                continue
            gid = re.search(r'gene_id "([^"]+)"', p[8]).group(1).split(".")[0]
            gname = re.search(r'gene_name "([^"]+)"', p[8]).group(1)
            rows.append((gid, gname, p[0], int(p[3]), int(p[4])))
    df = pd.DataFrame(rows, columns=["gene_id", "gene_name", "chromosome", "start", "end"])
    df.to_csv(cache, sep="\t", index=False)
    return df


def load_hallmark():
    if HALLMARK_CACHE.exists():
        return json.loads(HALLMARK_CACHE.read_text())
    import gseapy as gp
    lib = gp.get_library(name="MSigDB_Hallmark_2020", organism="Human")
    HALLMARK_CACHE.write_text(json.dumps(lib))
    return lib


def match_hallmark(lib, name):
    norm = lambda s: re.sub(r"[^a-z0-9]", "", s.lower())
    hit = [k for k in lib if norm(k) == norm(name)]
    if not hit:
        raise KeyError(f"Hallmark set '{name}' not found in library")
    return hit[0]


def cluster_means(adata, key, genes):
    genes = [g for g in genes if g in adata.var_names]
    codes, uniq = pd.factorize(adata.obs[key].astype(str))
    ind = sp.csr_matrix((np.ones(len(codes)), (codes, np.arange(len(codes)))),
                        shape=(len(uniq), len(codes)))
    ind = sp.diags(1.0 / np.bincount(codes)) @ ind
    X = adata[:, genes].X
    M = ind @ (X if sp.issparse(X) else sp.csr_matrix(X))
    return pd.DataFrame(np.asarray(M.todense()), index=uniq, columns=genes)


def score_and_label(adata, key, panels, overrides, resolve_ambiguous=False):
    """Per-cluster panel scores (mean of min-max scaled marker means) + guarded labels.
    resolve_ambiguous=True (myeloid states form a continuum): every cluster receives its best
    panel and the confidence (high / ambiguous / low) is reported instead of discarding cells."""
    allg = sorted({g for v in panels.values() for g in v})
    means = cluster_means(adata, key, allg)
    scaled = (means - means.min()) / (means.max() - means.min() + 1e-9)
    scores = pd.DataFrame({k: scaled[[g for g in v if g in scaled.columns]].mean(axis=1)
                           for k, v in panels.items()
                           if any(g in scaled.columns for g in v)})
    ordered = np.sort(scores.values, axis=1)
    top, second = ordered[:, -1], ordered[:, -2]
    label = scores.idxmax(axis=1).copy()
    low = top < ANNOT_MIN_TOP
    amb = (top - second < ANNOT_MARGIN) & (second > ANNOT_SECOND_HIGH)
    confidence = np.where(low, "low", np.where(amb, "ambiguous", "high"))
    if not resolve_ambiguous:
        label[low] = "Unassigned"
        label[amb] = "Ambiguous"
    for k, v in overrides.items():
        if str(k) in label.index:
            label[str(k)] = v
    table = scores.copy()
    table["top_score"], table["margin"], table["label"] = top, top - second, label
    table["second_best"] = scores.apply(lambda r: r.nlargest(2).index[-1], axis=1)
    table["confidence"] = confidence
    table["n_cells"] = adata.obs[key].astype(str).value_counts().reindex(table.index).values
    return label, table


def contamination_ratio(my, adata, key, max_ref_cells=20000):
    """Mean panel expression of each myeloid cluster divided by the level of that panel in
    bona fide cells of the lineage (whole atlas). Ratios > 0.5 flag doublet/contaminant clusters."""
    rng = np.random.default_rng(SEED)
    ref, lins = {}, [l for l in MAJOR_PANELS if l not in ("Myeloid", "Erythrocyte")]
    for l in lins:
        genes = [g for g in MAJOR_PANELS[l] if g in adata.var_names]
        idx = np.where(adata.obs.cell_type_major.astype(str).values == l)[0]
        if len(idx) < 50 or not genes:
            continue
        idx = rng.choice(idx, min(len(idx), max_ref_cells), replace=False)
        ref[l] = float(np.asarray(adata[idx][:, genes].X.mean(axis=0)).mean())
    means = cluster_means(my, key, sorted({g for l in ref for g in MAJOR_PANELS[l]}))
    return pd.DataFrame({l: means[[g for g in MAJOR_PANELS[l] if g in means.columns]].mean(axis=1) / ref[l]
                         for l in ref})


def pseudobulk(adata, keys, layer="counts"):
    grp = adata.obs[keys].astype(str).agg("|".join, axis=1) if isinstance(keys, list) \
        else adata.obs[keys].astype(str)
    codes, uniq = pd.factorize(grp)
    ind = sp.csr_matrix((np.ones(len(codes), dtype=np.float32), (codes, np.arange(len(codes)))),
                        shape=(len(uniq), len(codes)))
    X = adata.layers[layer] if layer else adata.X
    M = ind @ (X if sp.issparse(X) else sp.csr_matrix(X))
    return (pd.DataFrame(np.asarray(M.todense()), index=uniq, columns=adata.var_names),
            pd.Series(np.bincount(codes), index=uniq))


def aucell(X, gene_names, gene_sets, top_frac=0.05, chunk=2000, seed=SEED):
    """AUCell (Aibar et al., 2017): AUC of each gene set in the top 5% of each cell's ranking.
    Ties broken randomly (fixed gene permutation), as in the original implementation."""
    X = X.tocsr() if sp.issparse(X) else sp.csr_matrix(X)
    n_genes = X.shape[1]
    max_rank = int(np.ceil(top_frac * n_genes))
    rng = np.random.default_rng(seed)
    perm = rng.permutation(n_genes)
    members, n_eff = {}, {}
    gidx = pd.Index(gene_names)
    for name, genes in gene_sets.items():
        m = np.zeros(n_genes, dtype=np.float32)
        pos = gidx.get_indexer([g for g in genes if g in gidx])
        pos = pos[pos >= 0]
        m[pos] = 1.0
        members[name], n_eff[name] = m[perm], max(len(pos), 1)
    weights = (max_rank - np.arange(max_rank)).astype(np.float32)
    out = {k: np.zeros(X.shape[0], dtype=np.float32) for k in gene_sets}
    for s in range(0, X.shape[0], chunk):
        D = X[s:s + chunk].toarray()[:, perm]
        top_idx = np.argpartition(-D, max_rank - 1, axis=1)[:, :max_rank]
        order = np.take_along_axis(D, top_idx, axis=1)
        top_idx = np.take_along_axis(top_idx, np.argsort(-order, axis=1, kind="stable"), axis=1)
        for k, m in members.items():
            hits = m[top_idx]
            out[k][s:s + chunk] = hits @ weights / (min(n_eff[k], max_rank) * max_rank)
    return pd.DataFrame(out)


def clr(counts_df, pseudo=0.5):
    L = np.log(counts_df + pseudo)
    return L.sub(L.mean(axis=1), axis=0)


# ==============================================================================
# 6. S0 -- SAMPLE SHEET
# ==============================================================================
def build_sample_sheet():
    rows = []
    for f in sorted(RAW_150290.glob("*.raw_gene_bc_matrices.tar.gz")):
        m = re.match(r"(GSM\d+)_(Pat\d+)-([AB])\.raw_gene_bc_matrices\.tar\.gz", f.name)
        if not m:
            continue
        gsm, pat, ab = m.groups()
        if ab == "B" and pat in GSE150290_TUMOR:
            lauren, dx = GSE150290_TUMOR[pat]
            rows.append(dict(dataset="GSE150290", sample_id=f"{pat}-B", patient_id=pat, gsm=gsm,
                             tissue="Tumor", lauren=lauren, geo_diagnosis=dx,
                             chemistry="10x_3p_v2", file=str(f)))
        elif ab == "A" and pat in GSE150290_NORMAL_PATIENTS:
            rows.append(dict(dataset="GSE150290", sample_id=f"{pat}-A", patient_id=pat, gsm=gsm,
                             tissue="Normal", lauren=np.nan, geo_diagnosis=np.nan,
                             chemistry="10x_3p_v2", file=str(f)))
    for s, (gsm, pid, tissue) in GSE183904_SAMPLES.items():
        f = RAW_183904 / f"{gsm}_{s}.csv.gz"
        rows.append(dict(dataset="GSE183904", sample_id=s, patient_id=pid, gsm=gsm, tissue=tissue,
                         lauren=np.nan, geo_diagnosis=np.nan, chemistry="10x_5p", file=str(f)))
    sheet = pd.DataFrame(rows)
    missing = sheet[~sheet["file"].map(lambda p: Path(p).exists())]
    if len(missing):
        raise FileNotFoundError("Missing input files:\n" + missing[["dataset", "sample_id", "file"]].to_string())
    exp = {("GSE150290", "Tumor"): 23, ("GSE150290", "Normal"): 24,
           ("GSE183904", "Tumor"): 26, ("GSE183904", "Normal"): 10}
    got = sheet.groupby(["dataset", "tissue"]).size().to_dict()
    if got != exp:
        raise RuntimeError(f"Sample sheet counts {got} differ from expected {exp}")
    sheet.to_csv(TAB / "S0_sample_sheet.csv", index=False)
    log(f"S0 sample sheet: {got}")
    return sheet


# ==============================================================================
# 7. S1a -- LOAD + CELL CALLING
# ==============================================================================
def load_all(sheet):
    out = H5 / f"S1a_raw_merged_{GENE_TAG}.h5ad"
    if out.exists() and "load" not in FORCE:
        return sc.read_h5ad(out)
    gtf = load_gencode()
    id2name = dict(zip(gtf.gene_id, gtf.gene_name))
    per_ds, calling = {"GSE150290": [], "GSE183904": []}, []
    for _, r in sheet.iterrows():
        log(f"  loading {r.dataset} {r.sample_id}")
        if r.dataset == "GSE150290":
            M, bcs, sym_hg19, ens = read_10x_v2_tar(r.file)
            # hg19 (Cell Ranger 2.1.1) symbols are outdated (e.g. IGJ -> JCHAIN); Ensembl stable
            # IDs are build-independent, so map them to current GENCODE v44 symbols
            sym = np.array([id2name.get(e, s_) for e, s_ in zip(ens, sym_hg19)])
            tot = np.asarray(M.sum(axis=1)).ravel()
            keep, thr = ordmag_cell_calling(tot)
            M, bcs = M[keep], bcs[keep]
            calling.append(dict(sample_id=r.sample_id, barcodes_raw=len(tot),
                                umi_threshold=thr, cells_called=int(keep.sum())))
        else:
            M, bcs, sym = read_count_csv(r.file, id2name=id2name)
        M, genes = aggregate_duplicate_genes(M, sym)
        a = ad.AnnData(X=M.astype(np.float32),
                       obs=pd.DataFrame(index=[f"{r.sample_id}_{b}" for b in bcs]),
                       var=pd.DataFrame(index=genes))
        for c in ["dataset", "sample_id", "patient_id", "gsm", "tissue", "lauren",
                  "geo_diagnosis", "chemistry"]:
            a.obs[c] = r[c]
        a.obs_names_make_unique()
        per_ds[r.dataset].append(a)
    pd.DataFrame(calling).to_csv(TAB / "S1_GSE150290_cell_calling.csv", index=False)
    merged = [ad.concat(v, join="outer", fill_value=0) for v in per_ds.values()]
    common = merged[0].var_names.intersection(merged[1].var_names)
    cov = []
    for panel_set, pname in [(MAJOR_PANELS, "major"), (MYELOID_PANELS, "myeloid")]:
        for k, genes in panel_set.items():
            for g in genes:
                cov.append(dict(panel=pname, group=k, gene=g,
                                in_GSE150290=g in merged[0].var_names, in_GSE183904=g in merged[1].var_names))
    cov = pd.DataFrame(cov)
    cov.to_csv(TAB / "S1_marker_gene_coverage.csv", index=False)
    miss = cov[~(cov.in_GSE150290 & cov.in_GSE183904)]
    if len(miss):
        log(f"  marker genes absent from >=1 dataset (excluded from panels): "
            f"{miss[['gene', 'in_GSE150290', 'in_GSE183904']].to_dict('records')}")
    log(f"  genes: GSE150290={merged[0].n_vars}, GSE183904={merged[1].n_vars}, shared={len(common)}")
    adata = ad.concat([m[:, common] for m in merged], join="inner")
    adata.X = sp.csr_matrix(adata.X, dtype=np.float32)
    for c in ["dataset", "sample_id", "patient_id", "tissue", "chemistry"]:
        adata.obs[c] = adata.obs[c].astype("category")
    for c in ["lauren", "geo_diagnosis", "gsm"]:
        adata.obs[c] = adata.obs[c].astype(str)
    adata.write_h5ad(out, compression="gzip")
    log(f"S1a merged: {adata.n_obs:,} cells x {adata.n_vars:,} genes")
    return adata


# ==============================================================================
# 8. S1b -- QC + DOUBLETS
# ==============================================================================
def upper_mad_outlier(x, nmads):
    med = np.median(x)
    mad = np.median(np.abs(x - med)) * 1.4826
    return np.zeros(len(x), bool) if mad == 0 else x > med + nmads * mad


def qc_and_doublets(adata):
    out = H5 / f"S1b_qc_{GENE_TAG}.h5ad"
    if out.exists() and "qc" not in FORCE:
        return sc.read_h5ad(out)
    adata.var["mt"] = adata.var_names.str.startswith("MT-")
    adata.var["ribo"] = adata.var_names.str.match(r"^RP[SL]\d")
    adata.var["hb"] = adata.var_names.str.match(r"^HB[ABDEGMQZ]\d?$")
    sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo", "hb"], percent_top=None,
                               log1p=True, inplace=True)
    adata.obs["n_cells_before_qc"] = adata.obs.groupby("sample_id", observed=True)["sample_id"].transform("size")
    keep = np.zeros(adata.n_obs, bool)
    for s, idx in adata.obs.groupby("sample_id", observed=True).indices.items():
        o = adata.obs.iloc[idx]
        k = ((o.n_genes_by_counts >= QC_MIN_GENES) & (o.pct_counts_mt < QC_MAX_PCT_MT)).values
        k &= ~upper_mad_outlier(o.log1p_total_counts.values, QC_NMADS)
        k &= ~upper_mad_outlier(o.log1p_n_genes_by_counts.values, QC_NMADS)
        keep[idx] = k
    pre = adata.obs.groupby("sample_id", observed=True).size()
    adata = adata[keep].copy()

    adata.obs["doublet_score"], adata.obs["predicted_doublet"] = np.nan, False
    for s, idx in adata.obs.groupby("sample_id", observed=True).indices.items():
        if len(idx) < 100:
            continue
        sub = adata[idx].copy()
        try:
            sc.pp.scrublet(sub, random_state=SEED, verbose=False)
            adata.obs.iloc[idx, adata.obs.columns.get_loc("doublet_score")] = sub.obs["doublet_score"].values
            adata.obs.iloc[idx, adata.obs.columns.get_loc("predicted_doublet")] = \
                sub.obs["predicted_doublet"].astype(bool).values
        except Exception as e:
            log(f"  scrublet failed for {s}: {e}")
    adata = adata[~adata.obs["predicted_doublet"].astype(bool)].copy()
    post = adata.obs.groupby("sample_id", observed=True).size()
    small = post[post < QC_MIN_CELLS_PER_SAMPLE].index
    if len(small):
        log(f"  excluding samples with <{QC_MIN_CELLS_PER_SAMPLE} cells after QC: {list(small)}")
        adata = adata[~adata.obs.sample_id.isin(small)].copy()
    sc.pp.filter_genes(adata, min_cells=QC_MIN_CELLS_PER_GENE)
    qc_tab = pd.DataFrame({"cells_before_qc": pre, "cells_after_qc_doublets": post}).fillna(0).astype(int)
    qc_tab["excluded_small"] = qc_tab.index.isin(small)
    qc_tab.to_csv(TAB / "S1_qc_per_sample.csv")
    for c in ["sample_id", "patient_id", "dataset", "tissue"]:
        adata.obs[c] = adata.obs[c].astype(str).astype("category")
    adata.write_h5ad(out, compression="gzip")
    log(f"S1b QC: {adata.n_obs:,} cells, {adata.n_vars:,} genes, {adata.obs.sample_id.nunique()} samples")
    return adata


# ==============================================================================
# 9. S1c -- NORMALISATION, scVI INTEGRATION, MAJOR LINEAGES
# ==============================================================================
def select_hvg(adata, n_top, batch_key="dataset"):
    ok = ~adata.var_names.str.match(EXCLUDE_GENE_RE)
    tmp = adata[:, ok].copy()
    sc.pp.highly_variable_genes(tmp, n_top_genes=n_top, flavor="seurat_v3", layer="counts",
                                batch_key=batch_key, subset=False)
    adata.var["highly_variable"] = False
    adata.var.loc[tmp.var_names[tmp.var.highly_variable], "highly_variable"] = True


def sketch_indices(adata, cap, key="sample_id"):
    """Per-sample geometric sketching on a within-sample PCA of log-normalised HVGs.
    Within-sample PCA avoids batch structure steering the sketch."""
    from geosketch import gs
    keep = []
    for s_id, idx in adata.obs.groupby(key, observed=True).indices.items():
        idx = np.asarray(idx)
        if cap is None or len(idx) <= cap:
            keep.extend(idx.tolist())
            continue
        sub = adata[idx][:, adata.var.highly_variable.values].copy()
        sc.pp.scale(sub, max_value=10)
        sc.tl.pca(sub, n_comps=min(30, len(idx) - 1), random_state=SEED)
        sel = np.asarray(gs(sub.obsm["X_pca"], cap, replace=False, seed=SEED))
        keep.extend(idx[sel].tolist())
    return np.sort(np.asarray(keep, dtype=int))


def train_scvi(adata, batch_key, tag, sketch_cap=None):
    """Train scVI (template-paper architecture) on a sketch, infer latent space for ALL cells."""
    import scvi
    scvi.settings.seed = SEED
    hv = adata[:, adata.var.highly_variable.values].copy()
    train_idx = sketch_indices(adata, sketch_cap) if sketch_cap else np.arange(adata.n_obs)
    tr = hv[train_idx].copy()
    scvi.model.SCVI.setup_anndata(tr, layer="counts", batch_key=batch_key)
    model = scvi.model.SCVI(tr, **SCVI_KW)
    epochs = int(min(SCVI_MAX_EPOCHS, max(50, round(20000 / tr.n_obs * 400))))
    log(f"  scVI[{tag}]: training on {tr.n_obs:,}/{adata.n_obs:,} cells, {hv.n_vars:,} HVGs, <= {epochs} epochs")
    model.train(max_epochs=epochs, early_stopping=True, early_stopping_patience=15)
    model.save(str(H5 / f"scvi_{tag}_{PIPE_TAG}"), overwrite=True)
    model.history.get("elbo_train", pd.DataFrame()).to_csv(TAB / f"S1_scvi_{tag}_elbo.csv")
    latent = model.get_latent_representation(adata=hv, batch_size=4096)
    in_train = np.zeros(adata.n_obs, bool)
    in_train[train_idx] = True
    return latent, in_train


def integrate_and_annotate(adata):
    out = H5 / f"S1c_integrated_{PIPE_TAG}.h5ad"
    if out.exists() and "integrate" not in FORCE:
        return sc.read_h5ad(out)
    adata.layers["counts"] = adata.X.copy()
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    select_hvg(adata, N_HVG)
    adata.obsm["X_scVI"], adata.obs["in_scvi_training"] = train_scvi(
        adata, "sample_id", "atlas", sketch_cap=SKETCH_PER_SAMPLE)
    sc.pp.neighbors(adata, use_rep="X_scVI", n_neighbors=N_NEIGHBORS, random_state=SEED)
    sc.tl.umap(adata, min_dist=0.3, random_state=SEED)
    sc.tl.leiden(adata, resolution=LEIDEN_RES_MAJOR, flavor="igraph", n_iterations=2,
                 directed=False, key_added="leiden_major", random_state=SEED)
    label, table = score_and_label(adata, "leiden_major", MAJOR_PANELS, MANUAL_OVERRIDES_MAJOR)
    table["n_cells_in_scvi_training"] = adata.obs.groupby("leiden_major", observed=True)[
        "in_scvi_training"].sum().reindex(table.index).values
    table.to_csv(TAB / "S1_major_cluster_scores.csv")
    adata.obs["cell_type_major"] = adata.obs["leiden_major"].astype(str).map(label)
    drop = adata.obs.cell_type_major.isin(["Ambiguous", "Unassigned", "Erythrocyte"])
    log(f"  removed {drop.sum():,} cells in ambiguous/unassigned/erythrocyte clusters "
        f"(see tables/S1_major_cluster_scores.csv)")
    adata = adata[~drop].copy()
    sc.pp.neighbors(adata, use_rep="X_scVI", n_neighbors=N_NEIGHBORS, random_state=SEED)
    sc.tl.umap(adata, min_dist=0.3, random_state=SEED)
    adata.write_h5ad(out, compression="gzip")
    rep_tab = adata.obs.groupby(["dataset", "cell_type_major"], observed=True)["in_scvi_training"].agg(
        n_cells="size", n_in_training="sum")
    rep_tab["fraction_in_training"] = rep_tab.n_in_training / rep_tab.n_cells
    rep_tab.to_csv(TAB / "S1_sketch_representation_by_lineage.csv")
    log(f"S1c integrated: {adata.obs.cell_type_major.value_counts().to_dict()}")
    return adata


# ==============================================================================
# 10. S1d -- inferCNV -> MALIGNANT EPITHELIUM
# ==============================================================================
def run_infercnv(adata):
    out = H5 / f"S1d_cnv_{PIPE_TAG}.h5ad"
    if out.exists() and "cnv" not in FORCE:
        return sc.read_h5ad(out)
    import infercnvpy as cnv
    gtf = load_gencode()
    gp = gtf[gtf.chromosome.isin([f"chr{i}" for i in range(1, 23)])].drop_duplicates("gene_name").set_index("gene_name")
    ref_types = ["T_NK", "B", "Plasma", "Myeloid", "Mast", "Fibroblast", "Endothelial", "Pericyte_SMC"]
    adata.obs["cnv_score"], adata.obs["cnv_call"] = np.nan, "not_tested"
    thr_rows = []
    rng = np.random.default_rng(SEED)
    for ds in adata.obs.dataset.cat.categories:
        o = adata.obs
        epi = np.where((o.dataset == ds) & (o.cell_type_major == "Epithelial"))[0]
        ref = []
        for ct in ref_types:
            ix = np.where((o.dataset == ds) & (o.cell_type_major == ct))[0]
            ref.extend(rng.choice(ix, min(len(ix), CNV_REF_PER_LINEAGE), replace=False).tolist())
        sub = adata[np.concatenate([epi, np.array(ref, int)])].copy()
        common = sub.var_names.intersection(gp.index)
        sub = sub[:, common].copy()
        sub.var["chromosome"] = gp.loc[common, "chromosome"].values
        sub.var["start"] = gp.loc[common, "start"].values
        sub.var["end"] = gp.loc[common, "end"].values
        is_ref = np.r_[np.zeros(len(epi), bool), np.ones(len(ref), bool)]
        sub.obs["cnv_group"] = np.where(is_ref, "reference",
                                        np.where(sub.obs.tissue == "Tumor", "epi_tumor", "epi_normal"))
        log(f"  inferCNV {ds}: {len(epi):,} epithelial vs {len(ref):,} reference cells, {sub.n_vars:,} genes")
        cnv.tl.infercnv(sub, reference_key="cnv_group", reference_cat=["reference"],
                        window_size=CNV_WINDOW, n_jobs=N_JOBS)
        Xc = sub.obsm["X_cnv"]
        Xc = Xc.toarray() if sp.issparse(Xc) else np.asarray(Xc)
        sub.obs["cnv_score"] = np.mean(Xc ** 2, axis=1)
        grp = sub.obs.cnv_group
        if (grp == "epi_normal").sum() >= 200:
            thr, basis = np.quantile(sub.obs.cnv_score[grp == "epi_normal"], CNV_NORMAL_Q), "adjacent-normal epithelium"
        else:
            thr, basis = np.quantile(sub.obs.cnv_score[grp == "reference"], CNV_NORMAL_Q), "reference cells"
        cnv.tl.pca(sub)
        cnv.pp.neighbors(sub)
        cnv.tl.leiden(sub, resolution=1.0)
        et = grp == "epi_tumor"
        above = sub.obs.cnv_score > thr
        frac = above[et].groupby(sub.obs.cnv_leiden[et], observed=True).mean()
        cf = sub.obs.cnv_leiden.map(frac).astype(float)
        call = np.where(cf >= 0.7, True, np.where(cf <= 0.3, False, above))
        malignant = et & call.astype(bool)
        names = sub.obs_names[:len(epi)]
        adata.obs.loc[names, "cnv_score"] = sub.obs.cnv_score.values[:len(epi)]
        adata.obs.loc[names, "cnv_call"] = np.where(malignant.values[:len(epi)], "Malignant", "Non-malignant")
        thr_rows.append(dict(dataset=ds, threshold=thr, basis=basis, quantile=CNV_NORMAL_Q,
                             epi_tumor=int(et.sum()), epi_normal=int((grp == "epi_normal").sum()),
                             malignant=int(malignant.sum())))
        try:
            cnv.pl.chromosome_heatmap(sub, groupby="cnv_group", show=False, figsize=(7, 3.5))
            fig = plt.gcf()
            savefig(fig, f"FigS3_infercnv_heatmap_{ds}", FIG_SUPP)
        except Exception as e:
            log(f"  chromosome heatmap failed ({ds}): {e}")
        sub.obs[["cnv_group", "cnv_score", "cnv_leiden"]].to_csv(TAB / f"S1_infercnv_cells_{ds}.csv")
    pd.DataFrame(thr_rows).to_csv(TAB / "S1_infercnv_thresholds.csv", index=False)
    lin = adata.obs.cell_type_major.astype(str).copy()
    lin[(lin == "Epithelial") & (adata.obs.cnv_call == "Malignant")] = "Malignant"
    lin[lin == "Epithelial"] = "Epithelial_NM"
    adata.obs["lineage"] = pd.Categorical(lin)
    # adjacent-normal samples: only epithelium (inferCNV negative control) is retained
    drop = (adata.obs.tissue == "Normal") & (adata.obs.cell_type_major != "Epithelial")
    log(f"  removing {int(drop.sum()):,} non-epithelial cells from adjacent-normal samples "
        f"(no downstream role; epithelium kept as CNV control)")
    adata = adata[~drop.values].copy()
    tab = adata.obs[adata.obs.tissue == "Tumor"].groupby(["sample_id", "lineage"], observed=True).size().unstack(fill_value=0)
    tab.to_csv(TAB / "S1_lineage_counts_per_tumor.csv")
    adata.write_h5ad(out, compression="gzip")
    log(f"S1d inferCNV done: {pd.DataFrame(thr_rows).to_dict('records')}")
    return adata


# ==============================================================================
# 11. S1e -- MYELOID RE-INTEGRATION + STATES (tumour samples only)
# ==============================================================================
def build_myeloid(adata):
    out = H5 / f"S1e_myeloid_{PIPE_TAG}_{MY_BUILD_TAG}.h5ad"
    if out.exists() and "myeloid" not in FORCE:
        return sc.read_h5ad(out)
    my = adata[(adata.obs.tissue == "Tumor") & (adata.obs.lineage == "Myeloid")].copy()
    my.X = my.layers["counts"].copy()
    my.uns.pop("log1p", None)
    sc.pp.normalize_total(my, target_sum=1e4)
    sc.pp.log1p(my)
    select_hvg(my, N_HVG_MYELOID)
    my.obsm["X_scVI"], my.obs["in_scvi_training"] = train_scvi(
        my, "sample_id", "myeloid", sketch_cap=SKETCH_PER_SAMPLE_MYELOID)
    sc.pp.neighbors(my, use_rep="X_scVI", n_neighbors=N_NEIGHBORS, random_state=SEED)
    sc.tl.leiden(my, resolution=LEIDEN_RES_MYELOID, flavor="igraph", n_iterations=2,
                 directed=False, key_added="leiden_myeloid", random_state=SEED)
    ratio = contamination_ratio(my, adata, "leiden_myeloid")
    # ambient RNA raises every cluster equally; a doublet/contaminant cluster must stand out
    med = ratio.median(axis=0)
    flag = (ratio > CONTAM_RATIO) & ratio.ge(CONTAM_FOLD * med)
    out_tab = ratio.copy()
    out_tab["flagged"] = flag.any(axis=1)
    out_tab["flag_lineages"] = flag.apply(lambda r: ",".join(r.index[r]), axis=1)
    out_tab["n_cells"] = my.obs.leiden_myeloid.astype(str).value_counts().reindex(out_tab.index).values
    out_tab.to_csv(TAB / "S1_myeloid_contamination_ratio.csv")
    contam = out_tab.index[out_tab.flagged]
    log(f"  myeloid clusters removed as doublet/contaminant: "
        f"{out_tab.loc[contam, ['n_cells', 'flag_lineages']].to_dict('index')}")
    my = my[~my.obs.leiden_myeloid.astype(str).isin(contam)].copy()
    sc.pp.neighbors(my, use_rep="X_scVI", n_neighbors=N_NEIGHBORS, random_state=SEED)
    sc.tl.umap(my, min_dist=0.3, random_state=SEED)
    sc.tl.leiden(my, resolution=LEIDEN_RES_MYELOID, flavor="igraph", n_iterations=2,
                 directed=False, key_added="leiden_myeloid", random_state=SEED)
    lab, tab = score_and_label(my, "leiden_myeloid", MYELOID_PANELS, {},
                               resolve_ambiguous=True)
    tab.to_csv(TAB / "S1_myeloid_cluster_scores.csv")
    my.obs["myeloid_state"] = my.obs.leiden_myeloid.astype(str).map(lab)
    my.obs["state_confidence"] = my.obs.leiden_myeloid.astype(str).map(tab["confidence"])
    log(f"  myeloid label confidence (cells): {my.obs.state_confidence.value_counts().to_dict()} "
        f"-- review tables/S1_myeloid_cluster_scores.csv")
    order = [s for s in MYELOID_PANELS if s in set(my.obs.myeloid_state)]
    my.obs["myeloid_state"] = pd.Categorical(my.obs.myeloid_state, categories=order)
    comp = {s: c for c, ss in COMPARTMENTS.items() for s in ss}
    my.obs["compartment"] = my.obs.myeloid_state.astype(str).map(comp).fillna("Other")
    my.write_h5ad(out, compression="gzip")
    log(f"S1e myeloid: {my.n_obs:,} cells; {my.obs.myeloid_state.value_counts().to_dict()}")
    return my


def relabel_myeloid(my):
    """Apply reviewed cluster overrides on the frozen myeloid object (no re-clustering)."""
    out = H5 / f"S1e_myeloid_{PIPE_TAG}_{MY_LABEL_TAG}.h5ad"
    if out.exists() and "relabel" not in FORCE:
        return sc.read_h5ad(out)
    cl = my.obs["leiden_myeloid"].astype(str)
    sizes = cl.value_counts()
    for k, (state, n_exp) in MANUAL_OVERRIDES_MYELOID.items():
        if sizes.get(str(k), -1) != n_exp:
            raise RuntimeError(f"Cluster {k} has {sizes.get(str(k), 0)} cells, expected {n_exp}: "
                               "myeloid clustering changed -- re-review overrides before relabelling")
    auto = my.obs["myeloid_state"].astype(str)
    new = auto.copy()
    for k, (state, _) in MANUAL_OVERRIDES_MYELOID.items():
        new[cl == str(k)] = state
    my.obs["myeloid_state_auto"] = auto.values
    my.obs["myeloid_state"] = pd.Categorical(new.values,
                                             categories=[s for s in MYELOID_ORDER if s in set(new)])
    comp = {s: c for c, ss in COMPARTMENTS.items() for s in ss}
    my.obs["compartment"] = my.obs["myeloid_state"].astype(str).map(comp).fillna("Other")
    pd.crosstab(my.obs["myeloid_state_auto"], my.obs["myeloid_state"]).to_csv(
        TAB / "S1_myeloid_relabel_crosstab.csv")
    ov = pd.DataFrame([dict(cluster=k, new_state=v[0], n_cells=v[1],
                            auto_state=auto[cl == k].iloc[0]) for k, v in MANUAL_OVERRIDES_MYELOID.items()])
    ov.to_csv(TAB / "S1_myeloid_manual_overrides.csv", index=False)
    my.write_h5ad(out, compression="gzip")
    log(f"S1e relabelled: {my.obs.myeloid_state.value_counts().to_dict()}")
    return my


# ==============================================================================
# 12. S2 -- MULTICELLULAR PROGRAMME (MOFA+ on non-myeloid pseudobulk)
# ==============================================================================
def prepare_views(adata, samples=None):
    tum = adata[adata.obs.tissue == "Tumor"]
    if samples is not None:
        tum = tum[tum.obs.sample_id.isin(samples)]
    all_samples = sorted(tum.obs.sample_id.astype(str).unique())
    ds_of = tum.obs.drop_duplicates("sample_id").set_index("sample_id")["dataset"].astype(str)
    lin_pb, _ = pseudobulk(tum, "lineage")
    lin_cpm = lin_pb.div(lin_pb.sum(axis=1), axis=0) * 1e6
    views, amb_rows = {}, []
    for v in MCP_VIEWS:
        sub = tum[tum.obs.lineage == v]
        if sub.n_obs == 0 or v not in lin_cpm.index:
            continue
        own = lin_cpm.loc[v]
        other = lin_cpm.drop(index=v).max(axis=0)
        src = lin_cpm.drop(index=v).idxmax(axis=0)
        ambient = own < AMBIENT_MIN_RATIO * other
        for g in ambient.index[ambient]:
            amb_rows.append(dict(view=v, gene=g, own_cpm=own[g], max_other_cpm=other[g], source=src[g]))
        pb, n = pseudobulk(sub, "sample_id")
        pb = pb.loc[:, ~ambient.reindex(pb.columns).fillna(False).values]
        pb = pb.loc[n[n >= PB_MIN_CELLS].index]
        if len(pb) < MOFA_MIN_SAMPLES_VIEW:
            log(f"  view {v}: only {len(pb)} samples with >={PB_MIN_CELLS} cells -> dropped")
            continue
        pb = pb.loc[:, ~pb.columns.str.match(EXCLUDE_GENE_RE)]
        cpm = pb.div(pb.sum(axis=1), axis=0) * 1e6
        pb = pb.loc[:, (cpm >= 1).mean(axis=0) >= 0.5]
        logcpm = np.log2(pb.div(pb.sum(axis=1), axis=0) * 1e6 + 1)
        ds = ds_of.loc[logcpm.index]
        logcpm = logcpm - logcpm.groupby(ds.values).transform("mean")      # remove dataset means
        top = logcpm.var(axis=0).sort_values(ascending=False).index[:MOFA_N_FEATURES]
        views[v] = logcpm[top]
    if samples is None and amb_rows:
        amb = pd.DataFrame(amb_rows)
        amb.to_csv(TAB / f"S2_{MCP_TAG}_ambient_genes_removed.csv", index=False)
        log(f"  ambient-gene filter removed per view: {amb.groupby('view').size().to_dict()}")
    return views, all_samples


def run_mofa(views, samples, tag, n_factors=MOFA_FACTORS):
    from mofapy2.run.entry_point import entry_point
    import h5py
    present = pd.DataFrame({v: pd.Series(True, index=df.index) for v, df in views.items()}).reindex(samples).fillna(False)
    samples = list(present.index[present.sum(axis=1) >= 2])
    vnames = list(views)
    data = [[views[v].reindex(samples).to_numpy(dtype=float)] for v in vnames]
    ent = entry_point()
    ent.set_data_options(scale_views=True)
    ent.set_data_matrix(data, likelihoods=["gaussian"] * len(vnames), views_names=vnames,
                        groups_names=["tumours"], samples_names=[samples],
                        features_names=[[f"{v}|{g}" for g in views[v].columns] for v in vnames])
    ent.set_model_options(factors=n_factors, spikeslab_weights=True, ard_weights=True)
    ent.set_train_options(iter=MOFA_ITER, convergence_mode="slow", seed=SEED, verbose=False)
    ent.build()
    ent.run()
    fn = H5 / f"mofa_{tag}_{PIPE_TAG}_{MCP_TAG}.hdf5"
    ent.save(str(fn), save_data=False)
    with h5py.File(fn, "r") as f:
        Z = np.array(f["expectations"]["Z"]["tumours"])
        if Z.shape[0] != len(samples):
            Z = Z.T
        W = {}
        for v in vnames:
            w = np.array(f["expectations"]["W"][v])
            W[v] = pd.DataFrame(w.T if w.shape[0] != views[v].shape[1] else w,
                                index=views[v].columns)
        r2 = np.array(f["variance_explained"]["r2_per_factor"]["tumours"])
        if r2.shape[0] != len(vnames):
            r2 = r2.T
    K = Z.shape[1]
    fac = [f"Factor{k + 1}" for k in range(K)]
    Zdf = pd.DataFrame(Z, index=samples, columns=fac)
    r2df = pd.DataFrame(r2, index=vnames, columns=fac)
    if r2df.values.max() <= 1.0:
        r2df *= 100.0
    for v in W:
        W[v].columns = fac
    return Zdf, W, r2df


def derive_mcp(adata):
    out = TAB / f"S2_MCP_scores_{PIPE_TAG}_{MCP_TAG}_{AXIS_TAG}.csv"
    if out.exists() and "mofa" not in FORCE:
        return pd.read_csv(out, index_col=0)
    zf = TAB / f"S2_MOFA_all_factors_{MCP_TAG}.csv"
    wf = TAB / f"S2_MOFA_weights_{MCP_TAG}.csv"
    rf = TAB / f"S2_MOFA_R2_percent_{MCP_TAG}.csv"
    if zf.exists() and wf.exists() and rf.exists() and "mofa" not in FORCE:
        log(f"S2 reusing fitted MOFA+ model ({MCP_TAG})")
        Z = pd.read_csv(zf, index_col=0)
        w_all = pd.read_csv(wf, index_col=[0, 1])
        W = {v: w_all.xs(v, level=0) for v in w_all.index.get_level_values(0).unique()}
        r2 = pd.read_csv(rf, index_col=0)
    else:
        views, samples = prepare_views(adata)
        log(f"S2 MOFA views: { {v: d.shape for v, d in views.items()} }")
        Z, W, r2 = run_mofa(views, samples, "all")
        r2.to_csv(rf)
        Z.to_csv(zf)                                        # raw (un-oriented) factors
        pd.concat({v: w for v, w in W.items()}).to_csv(wf)
    active = (r2 >= MCP_R2_ACTIVE).sum(axis=0)
    score = r2.sum(axis=0)
    if MCP_AXIS_FACTOR:
        chosen, rule = MCP_AXIS_FACTOR, "factor screen: all factors x myeloid states, global BH-FDR"
    else:
        multi = active[active >= 2].index
        chosen = score[multi].idxmax() if len(multi) else score.idxmax()
        rule = "largest total R2 among factors active in >=2 views"
    # orientation (sign is arbitrary in MOFA; does not affect inference)
    hall = load_hallmark()
    gs = set(hall[match_hallmark(hall, AXIS_ORIENT_SET)])
    w = W[AXIS_ORIENT_VIEW][chosen]
    sign = 1.0 if (w[w.index.isin(gs)].mean() - w.mean()) > 0 else -1.0
    log(f"  axis {AXIS_LABEL} = {chosen} x {sign:+.0f} ({rule}); total R2 {score[chosen]:.1f}%, "
        f"active in {active[chosen]} views")
    tum = adata.obs[adata.obs.tissue == "Tumor"]
    frac = tum.groupby("sample_id", observed=True)["lineage"].apply(
        lambda x: x.isin(["Fibroblast", "Endothelial", "Pericyte_SMC"]).mean())
    meta = adata.obs.drop_duplicates("sample_id").set_index("sample_id")
    res = pd.DataFrame({"MCP": Z[chosen] * sign})
    res["MCP_z"] = (res.MCP - res.MCP.mean()) / res.MCP.std(ddof=1)
    for c in ["dataset", "patient_id", "lauren", "geo_diagnosis"]:
        res[c] = meta.loc[res.index, c].astype(str).values
    res["stromal_fraction"] = frac.reindex(res.index).values
    res.to_csv(out)
    pd.concat({v: W[v][chosen] * sign for v in W}).to_csv(TAB / f"S2_axis_weights_{MCP_TAG}_{AXIS_TAG}.csv")
    json.dump({"chosen_factor": chosen, "sign": sign, "rule": rule, "axis_label": AXIS_LABEL,
               "active_views": int(active[chosen]), "total_R2": float(score[chosen])},
              open(TAB / f"S2_MCP_selection_{MCP_TAG}_{AXIS_TAG}.json", "w"), indent=2)
    log("S2 axis derived")
    return res


def axis_diagnostics(my, mcp):
    """(i) screen of every MOFA factor x myeloid state (transparent axis selection);
    (ii) confounder checks of the chosen axis."""
    Z = pd.read_csv(TAB / f"S2_MOFA_all_factors_{MCP_TAG}.csv", index_col=0)
    ds = mcp["dataset"]
    counts = my.obs.groupby(["sample_id", "myeloid_state"], observed=False).size().unstack(fill_value=0)
    counts = counts.loc[counts.sum(axis=1) >= COMP_MIN_MYELOID]
    counts = counts.loc[counts.index.intersection(Z.index)]
    counts = counts.loc[:, (counts > 0).sum(axis=0) >= 5]
    clr_df = clr(counts)
    rows = []
    for f in Z.columns:
        z = Z.loc[counts.index, f]
        meta = pd.DataFrame({"dataset": ds.reindex(counts.index).values,
                             "z": ((z - z.mean()) / z.std(ddof=1)).values}, index=counts.index)
        for st in counts.columns:
            d = meta.assign(y=clr_df[st].values)
            fit = smf.ols("y ~ C(dataset) + z", data=d).fit()
            per = {f"beta_{k}": smf.ols("y ~ z", data=d[d.dataset == k]).fit().params["z"]
                   for k in d.dataset.unique() if (d.dataset == k).sum() >= 8}
            rows.append(dict(factor=f, state=st, beta=fit.params["z"], p=fit.pvalues["z"], n=len(d), **per))
    scr = pd.DataFrame(rows)
    scr["fdr_global"] = multipletests(scr.p, method="fdr_bh")[1]
    bcols = [c for c in scr.columns if c.startswith("beta_")]
    scr["same_sign_all_datasets"] = scr.apply(lambda r: all(np.sign(r[c]) == np.sign(r.beta) for c in bcols), axis=1)
    scr.sort_values("p").to_csv(TAB / f"S2_factor_screen_myeloid_{MCP_TAG}.csv", index=False)
    piv_b = scr.pivot(index="factor", columns="state", values="beta")
    piv_q = scr.pivot(index="factor", columns="state", values="fdr_global")
    fig, ax = plt.subplots(figsize=(4.8, 2.6))
    vmax = np.nanmax(np.abs(piv_b.values))
    im = ax.imshow(piv_b.values, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
    for i in range(piv_b.shape[0]):
        for j in range(piv_b.shape[1]):
            q = piv_q.values[i, j]
            if q < 0.1:
                ax.text(j, i, fdr_stars(q) if q < 0.05 else "\u2020", ha="center", va="center", fontsize=6)
    ax.set_xticks(range(piv_b.shape[1]))
    ax.set_xticklabels(piv_b.columns, rotation=90, fontsize=5.5)
    ax.set_yticks(range(piv_b.shape[0]))
    ax.set_yticklabels(piv_b.index, fontsize=5.5)
    plt.colorbar(im, ax=ax, shrink=0.7, label="\u03b2 (CLR per SD)")
    ax.set_title("MOFA+ factors vs myeloid states (global FDR; \u2020 <0.1)", fontsize=6.5)
    savefig(fig, "FigS4_factor_screen_heatmap", FIG_SUPP)
    # confounders of the chosen axis
    n_cells = my.obs.groupby("sample_id", observed=True).size()
    conf = [dict(check="stromal cell fraction (Spearman)",
                 value=stats.spearmanr(mcp.MCP, mcp.stromal_fraction, nan_policy="omit")[0]),
            dict(check="myeloid cells per tumour (Spearman)",
                 value=stats.spearmanr(mcp.MCP, n_cells.reindex(mcp.index), nan_policy="omit")[0])]
    g = mcp[mcp.dataset == "GSE150290"]
    for col, a, b in [("geo_diagnosis", "EGC", "AGC"), ("lauren", "Intestinal", "Diffuse")]:
        x, y = g.loc[g[col] == a, "MCP_z"], g.loc[g[col] == b, "MCP_z"]
        if len(x) >= 3 and len(y) >= 3:
            conf.append(dict(check=f"GSE150290 {a} vs {b} (Mann-Whitney P)",
                             value=stats.mannwhitneyu(x, y).pvalue,
                             detail=f"median {a}={x.median():.2f}, {b}={y.median():.2f}"))
    pd.DataFrame(conf).to_csv(TAB / f"S2_axis_confounders_{MCP_TAG}_{AXIS_TAG}.csv", index=False)
    log(f"  axis confounders: {pd.DataFrame(conf).round(3).to_dict('records')}")


def mcp_validation(adata):
    """Leave-one-dataset-out reproducibility + Hallmark ORA of the MCP weights.
    Separate checkpoint: runs from the saved pooled model, so the MOFA fit is not repeated."""
    rep_file = TAB / f"S2_MCP_reproducibility_{MCP_TAG}_{AXIS_TAG}.csv"
    if rep_file.exists() and "mofa" not in FORCE:
        return
    res = pd.read_csv(TAB / f"S2_MCP_scores_{PIPE_TAG}_{MCP_TAG}_{AXIS_TAG}.csv", index_col=0)
    sel = json.load(open(TAB / f"S2_MCP_selection_{MCP_TAG}_{AXIS_TAG}.json"))
    chosen, sign = sel["chosen_factor"], sel["sign"]
    w_all = pd.read_csv(TAB / f"S2_MOFA_weights_{MCP_TAG}.csv", index_col=[0, 1])
    W = {v: w_all.xs(v, level=0) * sign for v in w_all.index.get_level_values(0).unique()}
    views, _ = prepare_views(adata)
    # ---- leave-one-dataset-out reproducibility ----
    rows = []
    for ds in sorted(res.dataset.unique()):
        ds_samples = res.index[res.dataset == ds].tolist()
        v_ds = {v: d.loc[d.index.intersection(ds_samples)] for v, d in views.items()}
        v_ds = {v: d for v, d in v_ds.items() if len(d) >= 10}
        if len(v_ds) < 2:
            rows.append(dict(dataset=ds, note="too few samples/views"))
            continue
        Zd, Wd, _ = run_mofa(v_ds, ds_samples, f"only_{ds}")
        # (view, gene) MultiIndex: the same gene appears in several views
        wf = pd.concat({v: W[v][chosen] for v in v_ds})
        best, best_r = None, 0.0
        for f in Zd.columns:
            wd = pd.concat({v: Wd[v][f] for v in v_ds})
            r = np.corrcoef(wf.reindex(wd.index).values, wd.values)[0, 1]
            if np.isfinite(r) and abs(r) > abs(best_r):
                best, best_r = f, r
        common = Zd.index.intersection(res.index)
        rs = stats.spearmanr(Zd.loc[common, best] * np.sign(best_r), res.loc[common, "MCP"])[0]
        rows.append(dict(dataset=ds, matched_factor=best, weight_correlation=best_r,
                         abs_weight_correlation=abs(best_r), sample_score_spearman=rs,
                         n_samples=len(common), views=",".join(v_ds)))
        log(f"  MCP reproducibility {ds}: {best} |r_weights|={abs(best_r):.2f}, "
            f"rho_scores={rs:.2f} (n={len(common)})")
    pd.DataFrame(rows).to_csv(rep_file, index=False)
    # ---- Hallmark ORA on MCP weights ----
    import gseapy as gp
    hall = load_hallmark()
    ora = []
    for v, w in W.items():
        for direction, genes in [("positive", w[chosen].sort_values(ascending=False).index[:100]),
                                 ("negative", w[chosen].sort_values().index[:100])]:
            try:
                e = gp.enrich(gene_list=list(genes), gene_sets=hall, background=list(w.index),
                              outdir=None, cutoff=1.0, verbose=False).results
                e["view"], e["direction"] = v, direction
                ora.append(e)
            except Exception as ex:
                log(f"  ORA failed {v}/{direction}: {ex}")
    if ora:
        pd.concat(ora).to_csv(TAB / f"S2_MCP_weight_hallmark_ORA_{MCP_TAG}_{AXIS_TAG}.csv", index=False)
    log("S2 MCP validation done")


# ==============================================================================
# 13. FIGURE 1 PANELS
# ==============================================================================
def fig1A(adata, my):
    mask = (adata.obs.tissue == "Tumor").values
    tum = ad.AnnData(obs=adata.obs.loc[mask, ["lineage"]].copy(),
                     obsm={"X_umap": adata.obsm["X_umap"][mask]})
    set_palette(tum, "lineage", LINEAGE_PALETTE)
    set_palette(my, "myeloid_state", MYELOID_PALETTE)
    fig, axes = plt.subplots(1, 2, figsize=(7.1, 3.3), gridspec_kw={"wspace": 0.08})
    sc.pl.umap(tum, color="lineage", ax=axes[0], show=False, frameon=False, size=1.2,
               legend_loc="on data", legend_fontsize=6, legend_fontweight="bold",
               legend_fontoutline=1.5, title=f"Whole tumour ({tum.n_obs:,} cells)")
    sc.pl.umap(my, color="myeloid_state", ax=axes[1], show=False, frameon=False, size=4,
               legend_loc="on data", legend_fontsize=6, legend_fontweight="bold",
               legend_fontoutline=1.5, add_outline=True, outline_width=(0.25, 0.05),
               title=f"Myeloid compartment ({my.n_obs:,} cells)")
    for ax in axes:
        rasterize_points(ax)
        umap_axis_arrows(ax)
    panel_label(axes[0], "A", x=0.0)
    savefig(fig, "Fig1A_umap_whole_tumour_and_myeloid")


def fig1B(my):
    genes = [g for g in FIG1B_MARKERS if g in my.var_names]
    fig, axes = plt.subplots(2, 4, figsize=(7.1, 3.4), gridspec_kw={"wspace": 0.05, "hspace": 0.15})
    for ax, g in zip(axes.ravel(), genes):
        sc.pl.umap(my, color=g, ax=ax, show=False, frameon=False, size=3, cmap=MARKER_CMAP,
                   colorbar_loc=None, title=g, sort_order=True)
        ax.title.set_fontstyle("italic")
        rasterize_points(ax)
    for ax in axes.ravel()[len(genes):]:
        ax.axis("off")
    cax = fig.add_axes([0.92, 0.35, 0.008, 0.3])
    cb = mpl.colorbar.ColorbarBase(cax, cmap=mpl.colormaps[MARKER_CMAP], ticks=[0, 1])
    cb.ax.set_yticklabels(["Min", "Max"], fontsize=6)
    cb.outline.set_linewidth(0.4)
    panel_label(axes[0, 0], "B", x=0.0)
    savefig(fig, "Fig1B_marker_umaps")


def fig1C(my):
    var = {s: [g for g in (MYELOID_PANELS.get(s) or EXTRA_MARKERS.get(s, []))[:3] if g in my.var_names]
           for s in my.obs.myeloid_state.cat.categories}
    var = {k: v for k, v in var.items() if v}
    dp = sc.pl.DotPlot(my, var_names=var, groupby="myeloid_state", standard_scale="var",
                       figsize=(7.1, 2.9))
    dp.style(cmap="Reds", dot_edge_color="black", dot_edge_lw=0.3, largest_dot=45,
             dot_max=1.0, grid=False)
    dp.legend(colorbar_title="Mean expression\n(scaled)", size_title="Fraction of cells\nin group (%)",
              width=1.1)
    dp.make_figure()
    fig = getattr(dp, "fig", None) or plt.gcf()
    for ax in fig.axes:
        for lab in ax.get_xticklabels():
            lab.set_fontstyle("italic")
            lab.set_fontsize(5.5)
    fig.text(0.0, 1.0, "C", fontsize=10, fontweight="bold", va="bottom")
    savefig(fig, "Fig1C_marker_dotplot")


def composition_table(my, mcp):
    counts = my.obs.groupby(["sample_id", "myeloid_state"], observed=False).size().unstack(fill_value=0)
    counts = counts.loc[counts.sum(axis=1) >= COMP_MIN_MYELOID]
    counts = counts.loc[counts.index.intersection(mcp.index)]
    return counts


def fig1D(my, mcp):
    counts = composition_table(my, mcp)
    prop = counts.div(counts.sum(axis=1), axis=0) * 100
    order = mcp.loc[prop.index, "MCP"].sort_values(ascending=False).index
    prop = prop.loc[order]
    states = [s for s in prop.columns if prop[s].sum() > 0]
    h = max(3.0, 0.11 * len(prop) + 0.8)
    fig = plt.figure(figsize=(4.2, h))
    gs = gridspec.GridSpec(1, 3, width_ratios=[0.06, 0.06, 1], wspace=0.04)
    a0, a1, a2 = [fig.add_subplot(gs[i]) for i in range(3)]
    mz = mcp.loc[order, "MCP_z"].values[:, None]
    vmax = np.abs(mz).max()
    a0.imshow(mz, aspect="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    dcol = [mpl.colors.to_rgb(DATASET_PALETTE[d]) for d in mcp.loc[order, "dataset"]]
    a1.imshow(np.array(dcol)[:, None, :], aspect="auto")
    for a, t in [(a0, AXIS_LABEL), (a1, "Dataset")]:
        a.set_xticks([])
        a.set_title(t, fontsize=6, rotation=90, fontweight="normal")
    a0.set_yticks(range(len(order)))
    a0.set_yticklabels(order, fontsize=5)
    a1.set_yticks([])
    left = np.zeros(len(prop))
    for s in states:
        a2.barh(np.arange(len(prop)), prop[s].values, left=left, color=MYELOID_PALETTE.get(s, "#BBB"),
                height=0.85, edgecolor="none", label=s)
        left += prop[s].values
    a2.set_ylim(len(prop) - 0.5, -0.5)
    a2.set_yticks([])
    a2.set_xlim(0, 100)
    a2.set_xlabel("Myeloid cell-state proportion (%)")
    a2.set_title("Cell type proportions (%)", fontsize=7)
    for sp_ in ["top", "right"]:
        a2.spines[sp_].set_visible(False)
    handles = [Patch(color=MYELOID_PALETTE.get(s, "#BBB"), label=s) for s in states]
    handles += [Patch(color=c, label=d) for d, c in DATASET_PALETTE.items()]
    a2.legend(handles=handles, bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False,
              fontsize=5.5, handlelength=1.0, handleheight=0.8)
    panel_label(a0, "D", x=-2.0)
    savefig(fig, "Fig1D_composition_along_MCP")
    prop.to_csv(TAB / "Fig1D_state_proportions.csv")


def composition_stats(my, mcp):
    counts = composition_table(my, mcp)
    # NB: must not be named "C" -- patsy resolves C() in the formula from the caller's namespace
    clr_df = clr(counts)
    meta = mcp.loc[counts.index].copy()
    meta["MCP_z"] = (meta.MCP - meta.MCP.mean()) / meta.MCP.std(ddof=1)
    formula_rhs = "C(dataset) + MCP_z" if meta.dataset.nunique() > 1 else "MCP_z"
    rows = []
    for s in counts.columns:
        if (counts[s] > 0).sum() < 5:
            continue
        df = meta.assign(y=clr_df[s].values)
        fit = smf.ols(f"y ~ {formula_rhs}", data=df).fit()
        rows.append(dict(state=s, beta_clr_per_SD=fit.params["MCP_z"], se=fit.bse["MCP_z"],
                         p=fit.pvalues["MCP_z"], n_samples=len(df),
                         spearman_rho_prop=stats.spearmanr(meta.MCP, counts[s] / counts.sum(axis=1))[0]))
    res = pd.DataFrame(rows)
    res["fdr"] = multipletests(res.p, method="fdr_bh")[1]
    res.to_csv(TAB / f"Fig1E_composition_vs_axis_{MY_LABEL_TAG}_{MCP_TAG}_{AXIS_TAG}.csv", index=False)
    return counts, res


def composition_sensitivity(my, mcp):
    """Robustness of Fig1E: (i) all cells, (ii) high-confidence clusters only, (iii) each dataset
    separately, (iv) label-free per-tumour marker-panel scores (independent of cluster labels)."""
    def comp_test(obs, label, rhs):
        counts = obs.groupby(["sample_id", "myeloid_state"], observed=False).size().unstack(fill_value=0)
        counts = counts.loc[counts.sum(axis=1) >= COMP_MIN_MYELOID]
        counts = counts.loc[counts.index.intersection(mcp.index)]
        clr_df = clr(counts)
        meta = mcp.loc[counts.index].copy()
        meta["MCP_z"] = (meta.MCP - meta.MCP.mean()) / meta.MCP.std(ddof=1)
        rows = []
        for st in counts.columns:
            if (counts[st] > 0).sum() < 5:
                continue
            f = smf.ols(f"y ~ {rhs}", data=meta.assign(y=clr_df[st].values)).fit()
            rows.append(dict(analysis=label, state=st, beta=f.params["MCP_z"], p=f.pvalues["MCP_z"],
                             n=len(meta)))
        r = pd.DataFrame(rows)
        r["fdr"] = multipletests(r.p, method="fdr_bh")[1]
        return r

    o = my.obs
    res = [comp_test(o, "all cells", "C(dataset) + MCP_z"),
           comp_test(o[o.state_confidence == "high"], "high-confidence clusters only", "C(dataset) + MCP_z")]
    for ds in sorted(mcp.dataset.unique()):
        smp = mcp.index[mcp.dataset == ds]
        res.append(comp_test(o[o.sample_id.astype(str).isin(smp)], f"{ds} only", "MCP_z"))
    rows = []
    n = o.groupby("sample_id", observed=True).size()
    ok = n[n >= COMP_MIN_MYELOID].index
    for st in [x for x in MYELOID_ORDER if x in MYELOID_PANELS]:
        g = [x for x in MYELOID_PANELS[st] if x in my.var_names]
        sc.tl.score_genes(my, g, score_name="_tmp_score", random_state=SEED)
        per = my.obs.groupby("sample_id", observed=True)["_tmp_score"].mean()
        per = per.reindex(ok).reindex(mcp.index).dropna()
        meta = mcp.loc[per.index].copy()
        meta["MCP_z"] = (meta.MCP - meta.MCP.mean()) / meta.MCP.std(ddof=1)
        meta["y"] = (per - per.mean()) / per.std(ddof=1)
        f = smf.ols("y ~ C(dataset) + MCP_z", data=meta).fit()
        rows.append(dict(analysis="label-free panel score", state=st, beta=f.params["MCP_z"],
                         p=f.pvalues["MCP_z"], n=len(meta)))
    my.obs.drop(columns="_tmp_score", inplace=True)
    r = pd.DataFrame(rows)
    r["fdr"] = multipletests(r.p, method="fdr_bh")[1]
    res.append(r)
    out = pd.concat(res)
    out.to_csv(TAB / f"Fig1E_composition_sensitivity_{MY_LABEL_TAG}_{MCP_TAG}_{AXIS_TAG}.csv", index=False)
    log("  Fig1E sensitivity (beta per SD MCP):\n" +
        out.pivot_table(index="state", columns="analysis", values="beta").round(2).to_string())
    return out


def fig1E(my, mcp):
    counts, res = composition_stats(my, mcp)
    prop = counts.div(counts.sum(axis=1), axis=0)
    states = [s for s in res.state if prop[s].mean() >= 0.01]
    n = len(states)
    ncol = min(5, n)
    nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(7.1, 1.55 * nrow), squeeze=False,
                             gridspec_kw={"wspace": 0.45, "hspace": 0.7})
    x = mcp.loc[prop.index, "MCP_z"]
    for ax, s in zip(axes.ravel(), states):
        y = prop[s]
        for d, c in DATASET_PALETTE.items():
            m = mcp.loc[prop.index, "dataset"] == d
            ax.scatter(x[m], y[m], s=7, color=c, edgecolor="black", linewidth=0.2, zorder=3)
        b = np.polyfit(x, y, 1)
        xx = np.linspace(x.min(), x.max(), 50)
        ax.plot(xx, np.polyval(b, xx), color="black", lw=0.7)
        r = res.set_index("state").loc[s]
        ax.set_title(s.replace("_", " "), fontsize=6.5, color=MYELOID_PALETTE.get(s, "black"))
        ax.text(0.02, 0.97, f"{fdr_stars(r.fdr)}  FDR={r.fdr:.2g}", transform=ax.transAxes,
                fontsize=5.5, va="top")
        ax.set_xlabel(f"{AXIS_LABEL} (z)", fontsize=6)
        ax.set_ylabel("Frequency", fontsize=6)
        for sp_ in ["top", "right"]:
            ax.spines[sp_].set_visible(False)
    for ax in axes.ravel()[n:]:
        ax.axis("off")
    handles = [Line2D([0], [0], marker="o", ls="", markerfacecolor=c, markeredgecolor="black",
                      markeredgewidth=0.2, markersize=4, label=d) for d, c in DATASET_PALETTE.items()]
    fig.legend(handles=handles, loc="upper right", frameon=False, fontsize=6, bbox_to_anchor=(1.0, 1.04))
    panel_label(axes[0, 0], "E", x=-0.3)
    savefig(fig, "Fig1E_state_frequency_vs_MCP")


def fig1F(my):
    hall = load_hallmark()
    sets = {h: hall[match_hallmark(hall, h)] for h in FIG1F_HALLMARKS}
    X = my.X
    expressed = np.asarray((X > 0).sum(axis=0)).ravel() > 0
    auc = aucell(X[:, expressed], my.var_names[expressed], sets)
    auc.index = my.obs_names
    auc.to_csv(TAB / "Fig1F_AUCell_hallmark.csv")
    for h in sets:
        v = auc[h].values
        my.obs[f"AUC_{h}"] = (v - v.min()) / (v.max() - v.min() + 1e-12)
    fig, axes = plt.subplots(3, 3, figsize=(5.2, 5.0), gridspec_kw={"wspace": 0.05, "hspace": 0.25})
    for ax, h in zip(axes.ravel(), sets):
        sc.pl.umap(my, color=f"AUC_{h}", ax=ax, show=False, frameon=False, size=3,
                   cmap=HALLMARK_CMAP, colorbar_loc=None, sort_order=True,
                   title="HALLMARK_\n" + h.upper().replace(" ", "_").replace("-", "_"))
        ax.title.set_fontsize(6)
        ax.title.set_fontweight("normal")
        rasterize_points(ax)
    cax = fig.add_axes([0.35, 0.05, 0.3, 0.012])
    cb = mpl.colorbar.ColorbarBase(cax, cmap=HALLMARK_CMAP, orientation="horizontal", ticks=[0, 1])
    cb.ax.set_xticklabels(["Min", "Max"], fontsize=6)
    cb.outline.set_linewidth(0.4)
    panel_label(axes[0, 0], "F", x=0.0)
    savefig(fig, "Fig1F_hallmark_AUCell_umaps")
    med = auc.groupby(my.obs.myeloid_state.values, observed=True).median()
    medz = (med - med.mean()) / med.std(ddof=0)
    fig, ax = plt.subplots(figsize=(3.4, 2.6))
    im = ax.imshow(medz.T.values, cmap="RdBu_r", vmin=-2, vmax=2, aspect="auto")
    ax.set_xticks(range(len(medz)))
    ax.set_xticklabels(medz.index, rotation=90, fontsize=5.5)
    ax.set_yticks(range(medz.shape[1]))
    ax.set_yticklabels(medz.columns, fontsize=5.5)
    plt.colorbar(im, ax=ax, shrink=0.6, label="Median AUC (z)")
    savefig(fig, "FigS5_hallmark_median_AUC_by_state", FIG_SUPP)


def pydeseq2_continuous(counts, meta, terms, coef="MCP_z"):
    import pydeseq2
    from pydeseq2.dds import DeseqDataSet
    from pydeseq2.ds import DeseqStats
    major, minor = [int(x) for x in pydeseq2.__version__.split(".")[:2]]
    if (major, minor) < (0, 5):
        raise RuntimeError("pydeseq2 >= 0.5 is required (continuous covariates + contrast vectors)")
    dds = DeseqDataSet(counts=counts, metadata=meta, design="~ " + " + ".join(terms),
                       refit_cooks=True, quiet=True)
    dds.deseq2()
    cols = list(dds.obsm["design_matrix"].columns)
    if coef not in cols:
        raise KeyError(f"{coef} not in design columns {cols}")
    contrast = np.array([1.0 if c == coef else 0.0 for c in cols])
    ds = DeseqStats(dds, contrast=contrast, quiet=True)
    ds.summary()
    return ds.results_df.copy()


def de_along_mcp(my, mcp):
    out = TAB / f"Fig1G_H_DE_results_{PIPE_TAG}_{MY_LABEL_TAG}_{MCP_TAG}_{AXIS_TAG}.csv"
    if out.exists() and "de" not in FORCE:
        return pd.read_csv(out)
    allres, concord = [], []
    for comp in COMPARTMENTS:
        sub = my[my.obs.compartment == comp]
        pb, n = pseudobulk(sub, "sample_id")
        keep = n[(n >= DE_MIN_CELLS)].index.intersection(mcp.index)
        pb = pb.loc[keep]
        if len(pb) < DE_MIN_SAMPLES:
            log(f"  DE {comp}: only {len(pb)} samples -> skipped")
            continue
        meta = mcp.loc[keep, ["dataset", "MCP"]].copy()
        meta["MCP_z"] = (meta.MCP - meta.MCP.mean()) / meta.MCP.std(ddof=1)
        meta = meta[["dataset", "MCP_z"]]
        cnt = pb.round().astype(int)
        cnt = cnt.loc[:, (cnt >= 10).sum(axis=0) >= max(3, int(0.25 * len(cnt)))]
        terms = ["dataset", "MCP_z"] if meta.dataset.nunique() > 1 and meta.dataset.value_counts().min() >= 3 \
            else ["MCP_z"]
        log(f"  DE {comp}: {len(cnt)} samples ({meta.dataset.value_counts().to_dict()}), "
            f"{cnt.shape[1]} genes, design ~{' + '.join(terms)}")
        res = pydeseq2_continuous(cnt, meta, terms)
        res["compartment"], res["n_samples"] = comp, len(cnt)
        res["n_cells_median"] = int(n.loc[keep].median())
        allres.append(res.rename_axis("gene").reset_index())
        for ds in meta.dataset.unique():
            m = meta.dataset == ds
            if m.sum() < 6:
                continue
            md = meta.loc[m, ["MCP_z"]].copy()
            md["MCP_z"] = (md.MCP_z - md.MCP_z.mean()) / md.MCP_z.std(ddof=1)
            try:
                rd = pydeseq2_continuous(cnt.loc[m], md, ["MCP_z"])
                sig = res.index[(res.padj < DE_FDR) & (res.log2FoldChange.abs() >= DE_LFC)]
                g = sig.intersection(rd.index)
                concord.append(dict(compartment=comp, dataset=ds, n_samples=int(m.sum()),
                                    n_pooled_sig=len(sig),
                                    spearman_lfc_sig=stats.spearmanr(res.loc[g, "log2FoldChange"],
                                                                     rd.loc[g, "log2FoldChange"])[0] if len(g) > 5 else np.nan,
                                    sign_agreement_sig=np.mean(np.sign(res.loc[g, "log2FoldChange"])
                                                               == np.sign(rd.loc[g, "log2FoldChange"])) if len(g) else np.nan))
            except Exception as e:
                log(f"  per-dataset DE failed {comp}/{ds}: {e}")
    df = pd.concat(allres) if allres else pd.DataFrame()
    df.to_csv(out, index=False)
    pd.DataFrame(concord).to_csv(TAB / "FigS5_DE_per_dataset_concordance.csv", index=False)
    return df


def volcano(res, comp, letter, fname):
    from adjustText import adjust_text
    d = res.dropna(subset=["log2FoldChange", "padj"]).copy()
    d["nl"] = -np.log10(d.padj.clip(lower=1e-300))
    up = d[(d.padj < DE_FDR) & (d.log2FoldChange >= DE_LFC)]
    dn = d[(d.padj < DE_FDR) & (d.log2FoldChange <= -DE_LFC)]
    fig, ax = plt.subplots(figsize=(3.0, 3.1))
    ax.scatter(d.log2FoldChange, d.nl, s=3, color="#D3D3D3", lw=0, zorder=1, rasterized=True)
    ax.scatter(up.log2FoldChange, up.nl, s=5, color="#E31A1C", lw=0, zorder=2, rasterized=True)
    ax.scatter(dn.log2FoldChange, dn.nl, s=5, color="#1F3FB5", lw=0, zorder=2, rasterized=True)
    lab = pd.concat([up.nsmallest(12, "padj"), dn.nsmallest(12, "padj")])
    texts = [ax.text(r.log2FoldChange, r.nl, r.gene, fontsize=5, fontstyle="italic") for _, r in lab.iterrows()]
    if texts:
        adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle="-", color="black", lw=0.3))
    ax.axhline(-np.log10(DE_FDR), ls="--", color="grey", lw=0.5)
    for v in [-DE_LFC, DE_LFC]:
        ax.axvline(v, ls="--", color="grey", lw=0.5)
    lim = max(1.0, np.nanpercentile(d.log2FoldChange.abs(), 99.9) * 1.1)
    ax.set_xlim(-lim, lim)
    ax.set_xlabel(rf"$log_2$ Fold Change (per SD of {AXIS_LABEL})")
    ax.set_ylabel(r"$-log_{10}$FDR")
    kw = dict(xycoords="axes fraction", arrowprops=dict(arrowstyle="-|>", lw=0.8, color="black",
                                                        mutation_scale=7))
    ax.annotate("", xy=(0.02, 1.06), xytext=(0.47, 1.06), **kw)
    ax.annotate("", xy=(0.98, 1.06), xytext=(0.53, 1.06), **kw)
    ax.text(0.25, 1.09, f"Upregulated in\n{AXIS_LABEL}-low {comp}s", transform=ax.transAxes, ha="center",
            va="bottom", fontsize=5.5)
    ax.text(0.75, 1.09, f"Upregulated in\n{AXIS_LABEL}-high {comp}s", transform=ax.transAxes, ha="center",
            va="bottom", fontsize=5.5)
    ax.text(0.98, 0.02, f"n = {int(res.n_samples.iloc[0])} tumours", transform=ax.transAxes,
            ha="right", fontsize=5.5)
    panel_label(ax, letter, x=-0.12, y=1.25)
    savefig(fig, fname)


def gsea_bars(res, comp, letter, fname):
    import gseapy as gp
    hall = load_hallmark()
    rnk = res.dropna(subset=["stat"]).set_index("gene")["stat"].sort_values(ascending=False)
    rnk = rnk[~rnk.index.duplicated()]
    pre = gp.prerank(rnk=rnk.reset_index().rename(columns={"gene": 0, "stat": 1}), gene_sets=hall, min_size=15, max_size=500,
                     permutation_num=GSEA_PERM, seed=SEED, threads=N_JOBS, outdir=None, verbose=False)
    g = pre.res2d.copy()
    for c in ["NES", "FDR q-val", "NOM p-val"]:
        g[c] = pd.to_numeric(g[c], errors="coerce")
    g.to_csv(TAB / f"{fname}_table.csv", index=False)
    floor = 1.0 / (GSEA_PERM + 1)
    g["nl"] = -np.log10(g["FDR q-val"].clip(lower=floor))
    sig = g[g["FDR q-val"] < 0.05]
    show = pd.concat([sig[sig.NES > 0].nlargest(10, "NES"), sig[sig.NES < 0].nsmallest(10, "NES")])
    if show.empty:
        show = g.reindex(g.NES.abs().sort_values(ascending=False).index).head(10)
    show = show.sort_values("nl")
    fig, ax = plt.subplots(figsize=(3.2, 0.2 * len(show) + 0.9))
    cols = ["#E8604C" if v > 0 else "#6A7FD1" for v in show.NES]
    ax.barh(range(len(show)), show.nl, color=cols, height=0.7)
    ax.set_yticks(range(len(show)))
    ax.set_yticklabels(show.Term, fontsize=6)
    ax.set_xlabel(r"$-log_{10}$ (FDR)")
    ax.axvline(-np.log10(0.05), ls="--", lw=0.5, color="grey")
    ax.set_title("GSEA: MSigDB_Hallmark_2020", fontsize=7)
    for sp_ in ["top", "right"]:
        ax.spines[sp_].set_visible(False)
    ax.legend(handles=[Patch(color="#E8604C", label=f"Enriched in {AXIS_LABEL}-high {comp}s"),
                       Patch(color="#6A7FD1", label=f"Enriched in {AXIS_LABEL}-low {comp}s")],
              frameon=False, fontsize=5.5, loc="lower right")
    panel_label(ax, letter, x=-0.9)
    savefig(fig, fname)


# ==============================================================================
# 14. SUPPLEMENTARY FIGURES
# ==============================================================================
def supp_qc(adata_qc):
    q = pd.read_csv(TAB / "S1_qc_per_sample.csv", index_col=0)
    # categorical -> plain string before reindex/fillna (a new category cannot be filled in)
    ds = adata_qc.obs.drop_duplicates("sample_id").set_index("sample_id")["dataset"].astype(str)
    ds.index = ds.index.astype(str)
    q.index = q.index.astype(str)
    q["dataset"] = ds.reindex(q.index).fillna("excluded").astype(str).values
    q = q.sort_values(["dataset", "cells_after_qc_doublets"])
    fig, axes = plt.subplots(1, 3, figsize=(7.1, 3.0), gridspec_kw={"width_ratios": [2.2, 1, 1], "wspace": 0.5})
    ax = axes[0]
    ax.barh(range(len(q)), q.cells_before_qc, color="#D9D9D9", label="Before QC")
    ax.barh(range(len(q)), q.cells_after_qc_doublets,
            color=[DATASET_PALETTE.get(d, "#999") for d in q.dataset], label="After QC")
    ax.set_yticks(range(len(q)))
    ax.set_yticklabels(q.index, fontsize=3.5)
    ax.set_xlabel("Cells per sample")
    ax.legend(frameon=False)
    o = adata_qc.obs
    for ax, col, lab in [(axes[1], "n_genes_by_counts", "Genes per cell"),
                         (axes[2], "pct_counts_mt", "Mitochondrial reads (%)")]:
        data = [o.loc[o.dataset == d, col].values for d in DATASET_PALETTE]
        vp = ax.violinplot(data, showextrema=False)
        for b, c in zip(vp["bodies"], DATASET_PALETTE.values()):
            b.set_facecolor(c)
            b.set_alpha(0.8)
        ax.set_xticks([1, 2])
        ax.set_xticklabels(list(DATASET_PALETTE), rotation=30, fontsize=6)
        ax.set_ylabel(lab)
    savefig(fig, "FigS1_QC", FIG_SUPP)


def supp_integration(adata):
    from sklearn.metrics import silhouette_score
    rng = np.random.default_rng(SEED)
    idx = rng.choice(adata.n_obs, min(20000, adata.n_obs), replace=False)
    sub = adata[idx].copy()
    hv = sub[:, sub.var.highly_variable].copy()
    sc.pp.scale(hv, max_value=10)
    sc.tl.pca(hv, n_comps=20, random_state=SEED)
    rows = []
    for name, emb in [("PCA (uncorrected)", hv.obsm["X_pca"]), ("scVI", sub.obsm["X_scVI"])]:
        rows.append(dict(embedding=name,
                         silhouette_dataset=silhouette_score(emb, sub.obs.dataset.astype(str)),
                         silhouette_lineage=silhouette_score(emb, sub.obs.lineage.astype(str))))
    pd.DataFrame(rows).to_csv(TAB / "FigS2_integration_silhouette.csv", index=False)
    set_palette(adata, "dataset", DATASET_PALETTE)
    fig, axes = plt.subplots(1, 3, figsize=(7.1, 2.5), gridspec_kw={"wspace": 0.1})
    sc.pl.umap(adata, color="dataset", ax=axes[0], show=False, frameon=False, size=0.8, title="Dataset")
    sc.pl.umap(adata, color="tissue", ax=axes[1], show=False, frameon=False, size=0.8, title="Tissue",
               palette=["#9E9E9E", "#B2182B"])
    sc.pl.umap(adata, color="sample_id", ax=axes[2], show=False, frameon=False, size=0.8,
               title="Sample", legend_loc=None)
    for ax in axes:
        rasterize_points(ax)
        umap_axis_arrows(ax)
    savefig(fig, "FigS2_integration_umaps", FIG_SUPP)
    var = {k: [g for g in v if g in adata.var_names][:4] for k, v in MAJOR_PANELS.items()
           if k != "Erythrocyte"}
    var = {k: v for k, v in var.items() if v}
    sc.pl.dotplot(adata, var, groupby="lineage", standard_scale="var", cmap="Reds", show=False,
                  figsize=(7.1, 2.6))
    savefig(plt.gcf(), "FigS2_lineage_marker_dotplot", FIG_SUPP)


def supp_cnv(adata):
    o = adata.obs[adata.obs.cnv_call != "not_tested"]
    thr = pd.read_csv(TAB / "S1_infercnv_thresholds.csv").set_index("dataset")
    fig, axes = plt.subplots(1, len(thr), figsize=(3.3 * len(thr), 2.3), squeeze=False)
    for ax, ds in zip(axes.ravel(), thr.index):
        d = o[o.dataset == ds]
        for t, c in [("Normal", "#9E9E9E"), ("Tumor", "#B2182B")]:
            v = np.log10(d.loc[d.tissue == t, "cnv_score"].astype(float) + 1e-6)
            ax.hist(v, bins=60, histtype="step", color=c, density=True, lw=0.8,
                    label=f"Epithelium, {t.lower()} samples")
        ax.axvline(np.log10(thr.loc[ds, "threshold"] + 1e-6), color="black", ls="--", lw=0.6)
        ax.set_title(f"{ds}: {int(thr.loc[ds, 'malignant']):,} malignant cells", fontsize=7)
        ax.set_xlabel(r"$log_{10}$ CNV score")
        ax.set_ylabel("Density")
        ax.legend(frameon=False, fontsize=5)
    savefig(fig, "FigS3_cnv_score_distributions", FIG_SUPP)


def supp_mofa(mcp):
    r2 = pd.read_csv(TAB / f"S2_MOFA_R2_percent_{MCP_TAG}.csv", index_col=0)
    sel = json.load(open(TAB / f"S2_MCP_selection_{MCP_TAG}_{AXIS_TAG}.json"))
    rep = pd.read_csv(TAB / f"S2_MCP_reproducibility_{MCP_TAG}_{AXIS_TAG}.csv")
    fig, axes = plt.subplots(1, 3, figsize=(7.1, 2.4), gridspec_kw={"width_ratios": [1.6, 1, 1], "wspace": 0.6})
    im = axes[0].imshow(r2.values, cmap="Purples", aspect="auto")
    axes[0].set_xticks(range(r2.shape[1]))
    axes[0].set_xticklabels([c.replace("Factor", "F") for c in r2.columns], fontsize=5.5)
    axes[0].set_yticks(range(r2.shape[0]))
    axes[0].set_yticklabels(r2.index, fontsize=5.5)
    k = list(r2.columns).index(sel["chosen_factor"])
    axes[0].add_patch(plt.Rectangle((k - 0.5, -0.5), 1, r2.shape[0], fill=False, lw=1.0, ec="#E31A1C"))
    plt.colorbar(im, ax=axes[0], shrink=0.7, label="Variance explained (%)")
    axes[0].set_title(f"MOFA+ factors ({AXIS_LABEL} boxed)", fontsize=7)
    if "abs_weight_correlation" in rep:
        axes[1].bar(rep.dataset, rep.abs_weight_correlation, color=[DATASET_PALETTE.get(d) for d in rep.dataset])
        axes[1].set_ylim(0, 1)
        axes[1].set_ylabel("|r| weights, single-dataset\nvs pooled model")
        axes[1].tick_params(axis="x", rotation=30)
    g = mcp[mcp.dataset == "GSE150290"]
    groups = [("Intestinal", "IGC"), ("Diffuse", "DGC")]
    data = [g.loc[g.lauren == k, "MCP_z"].values for k, _ in groups]
    axes[2].boxplot(data, widths=0.5, showfliers=False, medianprops=dict(color="black", lw=0.8))
    for i, d in enumerate(data, 1):
        axes[2].scatter(np.full(len(d), i) + np.random.default_rng(SEED).uniform(-0.12, 0.12, len(d)),
                        d, s=6, color=DATASET_PALETTE["GSE150290"], edgecolor="black", lw=0.2, zorder=3)
    if all(len(d) >= 3 for d in data):
        p = stats.mannwhitneyu(*data).pvalue
        axes[2].set_title(f"Post hoc: Lauren (GSE150290)\nMann-Whitney P = {p:.2g}", fontsize=6)
    axes[2].set_xticks([1, 2])
    axes[2].set_xticklabels([l for _, l in groups])
    axes[2].set_ylabel(f"{AXIS_LABEL} (z)")
    savefig(fig, "FigS4_MOFA_MCP_summary", FIG_SUPP)


def supp_myeloid_evidence(my):
    tab = pd.read_csv(TAB / "S1_myeloid_cluster_scores.csv", index_col=0)
    sc_cols = [c for c in tab.columns if c in MYELOID_PANELS]
    fig, ax = plt.subplots(figsize=(3.8, 0.16 * len(tab) + 1.0))
    im = ax.imshow(tab[sc_cols].values, cmap="Reds", aspect="auto", vmin=0, vmax=1)
    ax.set_xticks(range(len(sc_cols)))
    ax.set_xticklabels(sc_cols, rotation=90, fontsize=5.5)
    ax.set_yticks(range(len(tab)))
    ax.set_yticklabels([f"{i} -> {l}" for i, l in zip(tab.index, tab.label)], fontsize=5)
    plt.colorbar(im, ax=ax, shrink=0.5, label="Panel score")
    savefig(fig, "FigS5_myeloid_cluster_panel_scores", FIG_SUPP)


# ==============================================================================
# 15. METHODS SUMMARY (numbers for the manuscript)
# ==============================================================================
def write_methods_summary(adata, my, mcp):
    thr = pd.read_csv(TAB / "S1_infercnv_thresholds.csv")
    sel = json.load(open(TAB / f"S2_MCP_selection_{MCP_TAG}_{AXIS_TAG}.json"))
    lines = [
        f"Tumours analysed: {adata.obs[adata.obs.tissue == 'Tumor'].sample_id.nunique()} "
        f"({adata.obs[adata.obs.tissue == 'Tumor'].groupby('dataset', observed=True).sample_id.nunique().to_dict()})",
        f"Cells after QC/doublets/annotation: {adata.n_obs:,}; tumour cells: {(adata.obs.tissue == 'Tumor').sum():,}",
        f"Myeloid cells (tumour): {my.n_obs:,}; states: {my.obs.myeloid_state.value_counts().to_dict()}",
        f"inferCNV thresholds: {thr.to_dict('records')}",
        f"MCP: {sel}",
        f"HVGs: atlas {N_HVG}, myeloid {N_HVG_MYELOID} (technical + IG/TR V-segment genes excluded)",
        f"scVI training sketch: <= {SKETCH_PER_SAMPLE} cells/sample (geosketch); latent inferred for all "
        f"{adata.n_obs:,} retained cells; myeloid model trained on all {my.n_obs:,} myeloid cells",
        f"Samples with MCP: {len(mcp)}",
        f"Software: scanpy {sc.__version__}, anndata {ad.__version__}",
    ]
    for pkg in ["scvi", "infercnvpy", "mofapy2", "pydeseq2", "gseapy", "statsmodels"]:
        try:
            lines.append(f"  {pkg} {importlib.import_module(pkg).__version__}")
        except Exception:
            pass
    (TAB / "methods_numbers.txt").write_text("\n".join(lines))
    log("\n".join(lines))


# ==============================================================================
# 16. MAIN
# ==============================================================================
def main():
    np.random.seed(SEED)
    log("=" * 70)
    log(f"Run started (font: {FONT})")
    s1d = H5 / f"S1d_cnv_{PIPE_TAG}.h5ad"
    if s1d.exists() and not (FORCE & {"load", "qc", "integrate", "cnv"}):
        # resume: S1a-S1d (load, QC, scVI, annotation, inferCNV) and their supp figures are done
        log(f"Resuming from checkpoint {s1d.name}")
        adata = sc.read_h5ad(s1d)
    else:
        sheet = build_sample_sheet()
        adata = load_all(sheet)
        adata = qc_and_doublets(adata)
        supp_qc(adata)
        adata = integrate_and_annotate(adata)
        adata = run_infercnv(adata)
        supp_integration(adata)
        supp_cnv(adata)
    my = build_myeloid(adata)
    my = relabel_myeloid(my)
    mcp = derive_mcp(adata)
    mcp_validation(adata)
    axis_diagnostics(my, mcp)
    supp_mofa(mcp)

    fig1A(adata, my)
    fig1B(my)
    fig1C(my)
    fig1D(my, mcp)
    fig1E(my, mcp)
    composition_sensitivity(my, mcp)
    fig1F(my)
    de = de_along_mcp(my, mcp)
    letters = {"Monocyte-derived TAM": ("G", "I"), "Resident-like TAM": ("H", "J")}
    for comp, (lv, lg) in letters.items():
        r = de[de.compartment == comp] if len(de) else pd.DataFrame()
        if r.empty:
            log(f"  no DE results for {comp}")
            continue
        tag = comp.replace(" ", "_").replace("-", "")
        volcano(r, comp, lv, f"Fig1{lv}_volcano_{tag}")
        gsea_bars(r, comp, lg, f"Fig1{lg}_GSEA_{tag}")
    supp_myeloid_evidence(my)
    my.write_h5ad(H5 / f"S1e_myeloid_final_{PIPE_TAG}_{MY_LABEL_TAG}_{MCP_TAG}_{AXIS_TAG}.h5ad", compression="gzip")
    write_methods_summary(adata, my, mcp)
    shutil.rmtree(TMP, ignore_errors=True)
    log("DONE. Review tables/S1_major_cluster_scores.csv and S1_myeloid_cluster_scores.csv; "
        "set MANUAL_OVERRIDES_* if any cluster label needs correction, then re-run with "
        "FORCE={'integrate','cnv','myeloid','mofa','de'} as appropriate.")


if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python
# coding: utf-8
"""
GC_Fig1_redraw_layout.py -- layout-only redraw of FigS2 (integration UMAPs), Fig1E, Fig1F,
Fig1G/H (volcanoes) and Fig1I/J (GSEA).

NOTHING is recomputed: every number comes from the saved checkpoints/tables of
GC_TAM_Stage1_Figure1.py (UMAP coordinates, composition statistics, AUCell scores, DESeq2 and
GSEA tables). Existing figure files are overwritten under the same names (pdf/svg/png).
"""
import re
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# ------------------------------------------------------------------ paths / tags (as in main run)
import os
OUT = Path(os.environ.get("GC_BASE", "/home/arshia/payanname")) / "GC_Fig1_outputs"
H5, TAB = OUT / "h5ad", OUT / "tables"
FIG_MAIN, FIG_SUPP = OUT / "figures" / "main", OUT / "figures" / "supp"
PIPE_TAG, MY_LABEL_TAG, MCP_TAG, AXIS_TAG, AXIS_LABEL = "v3", "m3", "amb20", "F4", "SAP"
COMP_MIN_MYELOID, DE_FDR, DE_LFC, GSEA_PERM = 30, 0.05, 0.5, 1000

# ------------------------------------------------------------------ style (identical to main run)
def _pick_font():
    from matplotlib import font_manager
    avail = {f.name for f in font_manager.fontManager.ttflist}
    for f in ["Arial", "Helvetica", "Liberation Sans", "Nimbus Sans", "DejaVu Sans"]:
        if f in avail:
            return f
    return "DejaVu Sans"


mpl.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": [_pick_font()],
    "font.size": 7, "axes.titlesize": 8, "axes.titleweight": "bold",
    "axes.labelsize": 7, "xtick.labelsize": 6, "ytick.labelsize": 6,
    "legend.fontsize": 6, "legend.title_fontsize": 6.5,
    "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "xtick.major.size": 2.5, "ytick.major.size": 2.5, "lines.linewidth": 0.8,
    "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",
    "savefig.dpi": 600, "figure.dpi": 150, "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
})
MYELOID_PALETTE = {
    "TAM_C1QC": "#6A3D9A", "TAM_FOLR2": "#33A02C", "TAM_SPP1": "#E31A1C",
    "TAM_Inflammatory": "#FF9E4A", "TAM_IFN": "#F2D400", "TAM_Prolif": "#1A1A1A",
    "TAM_Transitional": "#A6761D",
    "Mono_classical": "#8C2D19", "Mono_nonclassical": "#C98B63", "cDC1": "#1F5FA8",
    "cDC2": "#4F8FD6", "mregDC": "#9ECAE9", "pDC": "#8C8C8C", "Neutrophil": "#BCBD22",
}
DATASET_PALETTE = {"GSE183904": "#4A7EB5", "GSE150290": "#E3893F"}
TISSUE_PALETTE = {"Normal": "#9E9E9E", "Tumor": "#B2182B"}
HALLMARK_CMAP = LinearSegmentedColormap.from_list(
    "hallmark", ["#FFFBEA", "#FDD5A0", "#F07C6C", "#B8337A", "#4B0C5E"])


def savefig(fig, name, folder):
    for ext in ["pdf", "svg", "png"]:
        fig.savefig(folder / f"{name}.{ext}")
    plt.close(fig)
    print(f"  replaced {folder.name}/{name}")


def panel_label(ax, letter, x=-0.02, y=1.02):
    ax.text(x, y, letter, fontsize=10, fontweight="bold", va="bottom", ha="right", transform=ax.transAxes)


def rasterize_points(ax):
    for coll in ax.collections:
        coll.set_rasterized(True)


def umap_axis_arrows(ax, label="UMAP"):
    kw = dict(xycoords="axes fraction", textcoords="axes fraction",
              arrowprops=dict(arrowstyle="-|>", lw=0.6, color="black", mutation_scale=6))
    ax.annotate("", xy=(0.16, 0.02), xytext=(0.02, 0.02), **kw)
    ax.annotate("", xy=(0.02, 0.16), xytext=(0.02, 0.02), **kw)
    ax.text(0.02, -0.02, f"{label} 1", transform=ax.transAxes, fontsize=5.5, va="top", ha="left")
    ax.text(-0.01, 0.02, f"{label} 2", transform=ax.transAxes, fontsize=5.5, va="bottom",
            ha="right", rotation=90)


def set_palette(adata, key, palette):
    adata.obs[key] = adata.obs[key].astype("category")
    adata.uns[f"{key}_colors"] = [palette.get(c, "#BBBBBB") for c in adata.obs[key].cat.categories]


def fdr_stars(q):
    return "***" if q < 0.001 else "**" if q < 0.01 else "*" if q < 0.05 else "ns"


def small_dot_legend(ax, palette, y=-0.13):
    """Compact legend placed BELOW the axes (small markers, no overlap with the embedding)."""
    handles = [Line2D([0], [0], marker="o", ls="", markerfacecolor=c, markeredgecolor="none",
                      markersize=3.5, label=k) for k, c in palette.items()]
    ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, y), ncol=len(handles),
              frameon=False, fontsize=5.5, handletextpad=0.2, columnspacing=1.0, borderaxespad=0)


# ================================================================== FigS2 -- integration UMAPs
def redraw_figS2():
    full = sc.read_h5ad(H5 / f"S1d_cnv_{PIPE_TAG}.h5ad", backed="r")
    obs = full.obs[["dataset", "tissue", "sample_id"]].copy()
    lite = ad.AnnData(obs=obs, obsm={"X_umap": np.asarray(full.obsm["X_umap"])})
    full.file.close()
    set_palette(lite, "dataset", DATASET_PALETTE)
    set_palette(lite, "tissue", TISSUE_PALETTE)
    fig, axes = plt.subplots(1, 3, figsize=(7.1, 2.7), gridspec_kw={"wspace": 0.1})
    sc.pl.umap(lite, color="dataset", ax=axes[0], show=False, frameon=False, size=0.8,
               title="Dataset", legend_loc=None)
    sc.pl.umap(lite, color="tissue", ax=axes[1], show=False, frameon=False, size=0.8,
               title="Tissue", legend_loc=None)
    sc.pl.umap(lite, color="sample_id", ax=axes[2], show=False, frameon=False, size=0.8,
               title="Sample", legend_loc=None)
    for ax in axes:
        rasterize_points(ax)
        umap_axis_arrows(ax)
    small_dot_legend(axes[0], DATASET_PALETTE)
    small_dot_legend(axes[1], {k: v for k, v in TISSUE_PALETTE.items()
                               if k in set(lite.obs.tissue.astype(str))})
    axes[2].text(0.5, -0.13, f"{lite.obs.sample_id.nunique()} samples", transform=axes[2].transAxes,
                 ha="center", va="top", fontsize=5.5)
    savefig(fig, "FigS2_integration_umaps", FIG_SUPP)


# ================================================================== Fig1E -- frequency vs axis
def redraw_fig1E(my, mcp):
    res = pd.read_csv(TAB / f"Fig1E_composition_vs_axis_{MY_LABEL_TAG}_{MCP_TAG}_{AXIS_TAG}.csv")
    counts = my.obs.groupby(["sample_id", "myeloid_state"], observed=False).size().unstack(fill_value=0)
    counts = counts.loc[counts.sum(axis=1) >= COMP_MIN_MYELOID]
    counts = counts.loc[counts.index.intersection(mcp.index)]
    prop = counts.div(counts.sum(axis=1), axis=0)
    states = [s for s in res.state if s in prop.columns and prop[s].mean() >= 0.01]
    n = len(states)
    ncol = min(5, n)
    nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(7.1, 1.75 * nrow), squeeze=False,
                             gridspec_kw={"wspace": 0.5, "hspace": 1.05})
    x = mcp.loc[prop.index, "MCP_z"]
    rr = res.set_index("state")
    for ax, s in zip(axes.ravel(), states):
        y = prop[s]
        for d, c in DATASET_PALETTE.items():
            m = mcp.loc[prop.index, "dataset"] == d
            ax.scatter(x[m], y[m], s=7, color=c, edgecolor="black", linewidth=0.2, zorder=3)
        b = np.polyfit(x, y, 1)
        xx = np.linspace(x.min(), x.max(), 50)
        ax.plot(xx, np.polyval(b, xx), color="black", lw=0.7)
        q = rr.loc[s, "fdr"]
        # state name and statistics ABOVE the axes, never on the data
        ax.set_title(s.replace("_", " "), fontsize=6.5, color=MYELOID_PALETTE.get(s, "black"), pad=11)
        ax.text(0.5, 1.02, f"{fdr_stars(q)}  FDR = {q:.2g}", transform=ax.transAxes,
                ha="center", va="bottom", fontsize=5.5, fontweight="bold" if q < 0.05 else "normal")
        ax.set_xlabel(f"{AXIS_LABEL} (z)", fontsize=6)
        ax.set_ylabel("Frequency", fontsize=6)
        for sp_ in ["top", "right"]:
            ax.spines[sp_].set_visible(False)
    for ax in axes.ravel()[n:]:
        ax.axis("off")
    handles = [Line2D([0], [0], marker="o", ls="", markerfacecolor=c, markeredgecolor="black",
                      markeredgewidth=0.2, markersize=4, label=d) for d, c in DATASET_PALETTE.items()]
    fig.legend(handles=handles, loc="lower center", frameon=False, fontsize=6, ncol=2,
               bbox_to_anchor=(0.5, 1.0))
    panel_label(axes[0, 0], "E", x=-0.3, y=1.15)
    savefig(fig, "Fig1E_state_frequency_vs_MCP", FIG_MAIN)


# ================================================================== Fig1F -- Hallmark AUCell UMAPs
def hallmark_title(h, width=20):
    words = textwrap.wrap(h.upper().replace("-", " "), width)
    return "HALLMARK_\n" + "_\n".join(w.replace(" ", "_") for w in words)


def redraw_fig1F(my):
    auc = pd.read_csv(TAB / "Fig1F_AUCell_hallmark.csv", index_col=0).reindex(my.obs_names)
    sets = list(auc.columns)
    for h in sets:
        v = auc[h].values
        my.obs[f"AUC_{h}"] = (v - np.nanmin(v)) / (np.nanmax(v) - np.nanmin(v) + 1e-12)
    fig, axes = plt.subplots(3, 3, figsize=(5.6, 6.1), gridspec_kw={"wspace": 0.18, "hspace": 0.55})
    for ax, h in zip(axes.ravel(), sets):
        sc.pl.umap(my, color=f"AUC_{h}", ax=ax, show=False, frameon=False, size=3,
                   cmap=HALLMARK_CMAP, colorbar_loc=None, sort_order=True, title="")
        ax.set_title(hallmark_title(h), fontsize=5.5, fontweight="normal", linespacing=1.05, pad=3)
        rasterize_points(ax)
    for ax in axes.ravel()[len(sets):]:
        ax.axis("off")
    cax = fig.add_axes([0.35, 0.05, 0.3, 0.012])
    cb = mpl.colorbar.ColorbarBase(cax, cmap=HALLMARK_CMAP, orientation="horizontal", ticks=[0, 1])
    cb.ax.set_xticklabels(["Min", "Max"], fontsize=6)
    cb.outline.set_linewidth(0.4)
    panel_label(axes[0, 0], "F", x=0.0, y=1.25)
    savefig(fig, "Fig1F_hallmark_AUCell_umaps", FIG_MAIN)


# ================================================================== Fig1G/H -- volcanoes
def redraw_volcano(res, comp, letter, fname):
    from adjustText import adjust_text
    d = res.dropna(subset=["log2FoldChange", "padj"]).copy()
    d["nl"] = -np.log10(d.padj.clip(lower=1e-300))
    up = d[(d.padj < DE_FDR) & (d.log2FoldChange >= DE_LFC)]
    dn = d[(d.padj < DE_FDR) & (d.log2FoldChange <= -DE_LFC)]
    fig, ax = plt.subplots(figsize=(3.0, 3.1))
    ax.scatter(d.log2FoldChange, d.nl, s=3, color="#D3D3D3", lw=0, zorder=1, rasterized=True)
    ax.scatter(up.log2FoldChange, up.nl, s=5, color="#E31A1C", lw=0, zorder=2, rasterized=True)
    ax.scatter(dn.log2FoldChange, dn.nl, s=5, color="#1F3FB5", lw=0, zorder=2, rasterized=True)
    lab = pd.concat([up.nsmallest(12, "padj"), dn.nsmallest(12, "padj")])
    texts = [ax.text(r.log2FoldChange, r.nl, r.gene, fontsize=5, fontstyle="italic") for _, r in lab.iterrows()]
    if texts:
        adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle="-", color="black", lw=0.3))
    ax.axhline(-np.log10(DE_FDR), ls="--", color="grey", lw=0.5)
    for v in [-DE_LFC, DE_LFC]:
        ax.axvline(v, ls="--", color="grey", lw=0.5)
    lim = max(1.0, np.nanpercentile(d.log2FoldChange.abs(), 99.9) * 1.1)
    ax.set_xlim(-lim, lim)
    ax.set_xlabel(rf"$log_2$ Fold Change (per SD of {AXIS_LABEL})")
    ax.set_ylabel(r"$-log_{10}$FDR")
    # header: two short arrows with a clear gap, each label in 3 short lines centred on its arrow
    kw = dict(xycoords="axes fraction", arrowprops=dict(arrowstyle="-|>", lw=0.8, color="black",
                                                        mutation_scale=7))
    ax.annotate("", xy=(0.03, 1.05), xytext=(0.43, 1.05), **kw)
    ax.annotate("", xy=(0.97, 1.05), xytext=(0.57, 1.05), **kw)
    ax.text(0.23, 1.085, f"Upregulated in\n{AXIS_LABEL}-low\n{comp}s", transform=ax.transAxes,
            ha="center", va="bottom", fontsize=5.5, linespacing=1.15)
    ax.text(0.77, 1.085, f"Upregulated in\n{AXIS_LABEL}-high\n{comp}s", transform=ax.transAxes,
            ha="center", va="bottom", fontsize=5.5, linespacing=1.15)
    ax.text(0.98, 0.02, f"n = {int(res.n_samples.iloc[0])} tumours", transform=ax.transAxes,
            ha="right", fontsize=5.5)
    panel_label(ax, letter, x=-0.12, y=1.30)
    savefig(fig, fname, FIG_MAIN)


# ================================================================== Fig1I/J -- GSEA bars
def redraw_gsea(comp, letter, fname):
    g = pd.read_csv(TAB / f"{fname}_table.csv")
    for c in ["NES", "FDR q-val"]:
        g[c] = pd.to_numeric(g[c], errors="coerce")
    floor = 1.0 / (GSEA_PERM + 1)
    g["nl"] = -np.log10(g["FDR q-val"].clip(lower=floor))
    sig = g[g["FDR q-val"] < 0.05]
    show = pd.concat([sig[sig.NES > 0].nlargest(10, "NES"), sig[sig.NES < 0].nsmallest(10, "NES")])
    if show.empty:
        show = g.reindex(g.NES.abs().sort_values(ascending=False).index).head(10)
    show = show.sort_values("nl")
    fig, ax = plt.subplots(figsize=(3.2, 0.2 * len(show) + 0.9))
    cols = ["#E8604C" if v > 0 else "#6A7FD1" for v in show.NES]
    ax.barh(range(len(show)), show.nl, color=cols, height=0.7)
    ax.set_yticks(range(len(show)))
    ax.set_yticklabels(show.Term, fontsize=6)
    ax.set_xlabel(r"$-log_{10}$ (FDR)")
    ax.axvline(-np.log10(0.05), ls="--", lw=0.5, color="grey")
    ax.set_title("GSEA: MSigDB_Hallmark_2020", fontsize=7)
    for sp_ in ["top", "right"]:
        ax.spines[sp_].set_visible(False)
    # legend BELOW the x-axis label, outside the bars
    ax.legend(handles=[Patch(color="#E8604C", label=f"Enriched in {AXIS_LABEL}-high {comp}s"),
                       Patch(color="#6A7FD1", label=f"Enriched in {AXIS_LABEL}-low {comp}s")],
              frameon=False, fontsize=5.5, loc="upper center", bbox_to_anchor=(0.35, -0.45 / (0.2 * len(show) + 0.35)),
              ncol=1, handlelength=1.2)
    panel_label(ax, letter, x=-0.9)
    savefig(fig, fname, FIG_MAIN)


# ================================================================== run
if __name__ == "__main__":
    my = sc.read_h5ad(H5 / f"S1e_myeloid_{PIPE_TAG}_{MY_LABEL_TAG}.h5ad")
    mcp = pd.read_csv(TAB / f"S2_MCP_scores_{PIPE_TAG}_{MCP_TAG}_{AXIS_TAG}.csv", index_col=0)
    de = pd.read_csv(TAB / f"Fig1G_H_DE_results_{PIPE_TAG}_{MY_LABEL_TAG}_{MCP_TAG}_{AXIS_TAG}.csv")

    redraw_figS2()
    redraw_fig1E(my, mcp)
    redraw_fig1F(my)
    for comp, (lv, lg) in {"Monocyte-derived TAM": ("G", "I"), "Resident-like TAM": ("H", "J")}.items():
        tag = comp.replace(" ", "_").replace("-", "")
        r = de[de.compartment == comp]
        if len(r):
            redraw_volcano(r, comp, lv, f"Fig1{lv}_volcano_{tag}")
        if (TAB / f"Fig1{lg}_GSEA_{tag}_table.csv").exists():
            redraw_gsea(comp, lg, f"Fig1{lg}_GSEA_{tag}")
    print("Done -- figures redrawn from saved results only.")